# Reproducing Li, Daie, Svoboda & Druckmann (2016) from CRCNS alm-3

**Paper.** Li N, Daie K, Svoboda K, Druckmann S (2016). Robust neuronal dynamics in premotor cortex
during motor planning. *Nature* 532, 459-464.
[doi:10.1038/nature17643](https://doi.org/10.1038/nature17643). The claims file was written from the
accepted author manuscript in PubMed Central (PMC5081260): main text, Methods and the legends of
Figures 1 to 6 and Extended Data Figures 1 to 10.

**Data set.** CRCNS alm-3, [doi:10.6080/K0RB72JW](https://doi.org/10.6080/K0RB72JW), "Data and
simulations related to: Robust neuronal dynamics in premotor cortex during motor planning",
data description version 0.71 (April 19, 2018). Files are streamed from the public S3 bucket
`crcnsarchive`, under `alm-3/live/`. Version: `unversioned`. CRCNS data sets have no versions, and
the paper cites none, so the pinned version is the live copy, the only copy of the data files
(`alm-3/orig/` holds only documentation); the MD5 sum of each of the 91 session files is computed from the
bytes read and compared in the data check with the data set's `checksums.md5`, and printed, so
that a later change can be detected. The raw voltage traces (`data/RawVoltageTraces/`, 6 to 12 GB per session) are not used.

**Methods from the authors' code.** The data set holds the authors' MATLAB analysis scripts
(`alm-3/live/analyses_scripts/`), which its description says reproduce the paper's figures. The
claims file did not read them. Where the claims file leaves a detail open, this notebook takes it
from the Methods, and where the Methods leave it open, from these scripts, citing the file and
section; where the scripts disagree with the Methods (cd as a linear discriminant in place of
the mean difference), the Methods give the main result and the scripts a variant. The helper
module `alm3.py` says where each parameter comes from.

**Claims.**

| Claim | Tested here |
|---|---|
| units-and-sessions | tested |
| ed1-selective-fraction | tested |
| fig1d-late-delay-ipsilateral-bias | blocked: no condition ending at the go cue in the recording sessions; the behaviour-only sessions are not in the data set |
| fig1d-earlier-photoinhibition-minimal | tested, on the recording sessions of Experiment 1 |
| ed3-recovery-interval | blocked: behaviour-only sessions not in the data set |
| ed2b-silencing | tested |
| ed2d-spatial-extent | blocked: recordings in untrained mice not in the data set |
| ed2e-no-rebound | tested |
| fig1f-rate-recovery | tested |
| fig1g-selectivity-recovery | tested |
| ed4-return-to-trajectory | tested |
| ed5-photoactivation | blocked: photoactivation data not in the data set |
| fig2b-bilateral-behaviour | tested, on the recording sessions |
| ed6-bilateral-bias-inconsistent | tested |
| fig2-bilateral-recovery | tested |
| ed7-unilateral-4-spots-bilateral-1-spot | tested |
| ed7e-size-and-roc | tested |
| fig3-cd-roc | tested |
| ed8-roc-during-photoinhibition | tested |
| ed9b-cd-stability | tested |
| ed8a-mode-variance | tested |
| fig3-cd-predicts-movement | tested |
| fig3-other-modes | tested |
| ed9f-reaction-time | blocked: the lick trace shows no licks in any Experiment 2 session (data check) |
| fig5b-callosotomy-behaviour | tested, after the cut only (no sessions before the cut) |
| fig5-callosotomy-recovery | tested |
| ed10e-callosotomy-preparatory-activity | tested |
| models-monolithic-fail | tested in part: three of the five models are simulated |
| fig6-modular-models-robust | tested in part: the attractor and the integrator are simulated |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import alm3

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

In [ ]:
started = time.time()
files = alm3.list_files()
md5 = alm3.checksums()
names = alm3.session_names()
sessions = alm3.load_all(names, workers=3)
units = [u for s in sessions for u in alm3.reduce_units(s)]
meta = alm3.load_meta()
import resource
print(f"loaded {len(sessions)} sessions and {len(units)} units in {time.time() - started:.0f} s; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB "
      f"(each of the 3 loading processes holds one session file, about 1.5 GB at most)")

In [ ]:
SD = {s["name"]: s for s in sessions}
EXP = {s["name"]: alm3.experiment(s) for s in sessions}
CONDS = {"unilateral": [1, 2, 3], "exp2": [4, 5, 6], "exp2b": [4, 5, 7], "exp3": [7, 8, 9], "ccx": [10, 11, 12], "none": []}
N_SCREEN = {"unilateral": 5, "exp2": 10, "exp2b": 10, "exp3": 10, "ccx": 10, "none": 10}
trials = pd.concat([alm3.trial_table(s).assign(experiment=EXP[s["name"]]) for s in sessions], ignore_index=True)
info = pd.DataFrame([{k: u[k] for k in ["session", "mouse", "unit", "experiment", "cell_type", "spike_width_ms", "n_spikes"]} for u in units])

In [ ]:
read = sorted(k for k in files if k.startswith("alm-3/live/data/data_structure_files/") and k.endswith(".mat"))
print(f"Files read: {len(read)} session files, {sum(files[k] for k in read) / 1e9:.2f} GB in the bucket listing.")
print("Session files without an MD5 sum in checksums.md5:", sum(k.replace("alm-3/live/", "") not in md5 for k in read))
# MD5 sums computed from the bytes read (alm3.load_session), against those listed in checksums.md5.
md5_read = {s["name"]: s["md5"] for s in sessions}
key = lambda name: f"data/data_structure_files/data_structure_{name}.mat"
print("Session files whose computed MD5 differs from checksums.md5:",
      sum(md5_read[n] != md5.get(key(n)) for n in md5_read), "of", len(md5_read))
print("MD5 sums computed from the files read:")
for n in sorted(md5_read):
    print(" ", f"data_structure_{n}.mat", md5_read[n], "" if md5_read[n] == md5.get(key(n)) else "DIFFERS from checksums.md5")

In [ ]:
ses = pd.DataFrame([{
    "session": s["name"], "mouse": s["mouse"], "experiment": EXP[s["name"]],
    "line": alm3.mouse_line(meta[s["name"]]["gene_modification"]), "trials": len(s["trial_ids"]), "good": int(s["good"].sum()),
    "units": len(s["units"]),
    "stim_types": " ".join(str(int(x)) for x in sorted(np.unique(s["stim_type"][s["good"] & np.isfinite(s["stim_type"])]))),
} for s in sessions])
print(ses.groupby("experiment").agg(sessions=("session", "size"), mice=("mouse", "nunique"), units=("units", "sum"), good_trials=("good", "sum")).to_string())
print("\nSessions per mouse and experiment:")
print(pd.crosstab(ses.mouse, ses.experiment).to_string())
print("\nPhotostimulus types present (good trials), by experiment:")
print(ses.groupby(["experiment", "stim_types"]).size().to_string())
print("\nSessions with no good trials:", ses[ses.good == 0].session.tolist())
print("Trial ids are 1..n in every session:", all((s["trial_ids"] == np.arange(1, len(s["trial_ids"]) + 1)).all() for s in sessions))
print("Trial types:", sorted({tuple(s["trial_type_names"]) for s in sessions}))
LINE = ses.groupby("mouse").line.first().to_dict()
print("\nMouse lines from the meta data files (gene modification), by experiment group:")
print(ses.assign(group=np.where(ses.experiment == "ccx", "callosotomy", "control")).groupby(["group", "line"]).mouse.nunique().to_string())
print("Mice whose sessions disagree on the line:", [m for m, g in ses.groupby("mouse") if g.line.nunique() > 1])
print("Data description: 5 VGAT-ChR2-EYFP and 7 PV x ReaChR mice for electrophysiology; callosotomy 6 VGAT-ChR2-EYFP and 1 PV x ReaChR.")
print("Time units (2 = second):", sorted({s["time_unit"] for s in sessions}), sorted({u["time_unit"] for s in sessions for u in s["units"]}))

In [ ]:
g = trials[trials.good & trials.cue.notna()].copy()
g["sample"] = g.pole_out - g.pole_in
g["delay"] = g.cue - g.pole_out
print("Epoch durations (s) on good trials, by experiment, excluding lick-early trials:")
print(g[~g.lick_early].groupby("experiment")[["sample", "delay"]].describe(percentiles=[0.01, 0.5, 0.99]).round(3).to_string())
print("\nLick-early trials: delay duration quantiles", g[g.lick_early].delay.quantile([0.01, 0.5, 0.99]).round(2).tolist(),
      "; sample", g[g.lick_early]["sample"].quantile([0.01, 0.5, 0.99]).round(2).tolist())
print("\nOutcomes on good trials (fractions):")
print(g.assign(outcome=np.select([g.hit, g.err, g.nolick], ["hit", "err", "nolick"], "other")).groupby("experiment").outcome.value_counts(normalize=True).unstack().round(3).to_string())
print("Trials with both or no side:", int((g.side == 0).sum()))

In [ ]:
# Laser timing against the trial table: onset and offset of the laser power trace (above 5% of
# its maximum), relative to the sample onset, delay onset and cue, per photostimulus type.
gl = g[~g.lick_early].copy()
gl["on_from_delay"] = gl.laser_on - gl.pole_out
gl["off_from_cue"] = gl.laser_off - gl.cue
gl["on_from_sample"] = gl.laser_on - gl.pole_in
lt = gl.groupby("stim").agg(n=("trial", "size"), with_laser=("laser_max", lambda x: int(x.notna().sum())),
                            power_mW=("laser_max", "median"), on_from_sample=("on_from_sample", "median"),
                            on_from_delay=("on_from_delay", "median"), off_from_cue=("off_from_cue", "median"),
                            duration=("laser_off", lambda x: np.nan))
lt["duration"] = gl.groupby("stim").apply(lambda d: (d.laser_off - d.laser_on).median())
print(lt.round(3).to_string())
print("Control trials with laser power above 0:", int(gl[(gl.stim == 0)].laser_max.gt(0).sum()), "of", int((gl.stim == 0).sum()))

In [ ]:
# Clock check. (1) Spikes against the laser trace: the mean rate of putative pyramidal neurons in
# 20 ms bins around the laser onset measured on each trial's own laser trace, ipsilateral
# photoinhibition trials (types 2, 4 and 10, which start at delay onset) against control trials
# aligned to the same nominal time (delay onset + 4 ms). The spike times are aligned through the
# trial table's cue time, so a drop that starts at the laser onset shows that spikes, laser trace
# and trial table share one clock. (2) The same, per session, over its photostimulus types that
# include the left (recorded) ALM, that is all but the contralateral ones (5, 9, 11).
def onset_aligned(u, s, times):
    tr = u["trials"]
    st = s["stim_type"][tr]
    ok = u["stable"] & ~s["trial_types"]["LickEarly"][tr]
    res = {}
    for name, m in (("stim", ok & np.isin(st, [2, 4, 10])), ("control", ok & (st == 0))):
        idx = np.flatnonzero(m)
        if len(idx) < 3:
            return None
        if name == "stim":
            ref = (s["laser_on"] - s["cue"])[tr[idx]]
            ok2 = np.isfinite(ref)
            idx, ref = idx[ok2], ref[ok2]
        else:
            ref = (s["pole_out"] - s["cue"])[tr[idx]] + 0.004
        # counts in 20 ms bins at times relative to ref, from the 10 ms bins (rounded to 10 ms)
        t = np.round((ref[:, None] + times[None, :]) / 0.01) * 0.01
        cs = np.concatenate([np.zeros((len(idx), 1)), np.cumsum(u["binned"][idx], 1)], 1)
        a = np.clip(np.round((t - alm3.T0) / 0.01).astype(int), 0, alm3.NB10)
        b = np.clip(a + 2, 0, alm3.NB10)
        res[name] = (np.take_along_axis(cs, b, 1) - np.take_along_axis(cs, a, 1)).mean(0) / 0.02
    return res

tt_on = np.round(np.arange(-0.2, 0.3, 0.02), 3)
acc = {"stim": [], "control": []}
for u in units:
    s = SD[u["session"]]
    if u["cell_type"] != "pyramidal" or EXP[u["session"]] == "exp3":
        continue
    r = onset_aligned(u, s, tt_on)
    if r is None:
        continue
    for k in acc:
        acc[k].append(r[k])
on_tab = pd.DataFrame({"t_from_laser_onset_s": tt_on, "control": np.mean(acc["control"], 0), "photoinhibition": np.mean(acc["stim"], 0)})
on_tab["ratio"] = on_tab.photoinhibition / on_tab.control
print(f"{len(acc['stim'])} pyramidal neurons; mean rate (spikes/s) in 20 ms bins starting at each time from laser onset:")
print(on_tab.round(2).to_string(index=False))

sess_rows = []
for name in ses.session:
    s = SD[name]
    us = [u for u in units if u["session"] == name and u["cell_type"] == "pyramidal"]
    pre, post = [], []
    for u in us:
        tr = u["trials"]
        st = s["stim_type"][tr]
        ok = u["stable"] & ~s["trial_types"]["LickEarly"][tr] & np.isin(st, [1, 2, 3, -1, -2, -3, 4, 6, 7, 8, 10, 12]) & np.isfinite((s["laser_on"])[tr])
        idx = np.flatnonzero(ok)
        if len(idx) < 5:
            continue
        ref = (s["laser_on"] - s["cue"])[tr[idx]]
        cs = np.concatenate([np.zeros((len(idx), 1)), np.cumsum(u["binned"][idx], 1)], 1)
        a0 = np.round((ref - 0.2 - alm3.T0) / 0.01).astype(int)
        a1 = np.round((ref - alm3.T0) / 0.01).astype(int)
        a2 = np.round((ref + 0.2 - alm3.T0) / 0.01).astype(int)
        r = np.arange(len(idx))
        pre.append((cs[r, a1] - cs[r, a0]).mean() / 0.2)
        post.append((cs[r, a2] - cs[r, a1 + 2]).mean() / 0.18)
    if pre:
        sess_rows.append({"session": name, "experiment": EXP[name], "neurons": len(pre), "rate_before": np.mean(pre), "rate_after_onset": np.mean(post)})
clk = pd.DataFrame(sess_rows)
clk["ratio"] = clk.rate_after_onset / clk.rate_before
print("\nPer session (types including left ALM): pyramidal rate 0.2 s before laser onset against 0.02 to 0.2 s after it")
print(clk.ratio.describe().round(3).to_string())
print("Sessions with ratio above 0.5:", clk[clk.ratio > 0.5][["session", "experiment", "neurons", "ratio"]].round(2).to_dict("records"))
print("Sessions with photostimulation but no pyramidal neuron to check:", sorted(set(ses[ses.stim_types.str.len() > 1].session) - set(clk.session)))

In [ ]:
# The lick trace (channel "lick_trace", a photodiode signal). The first lick after the cue is the
# first upward crossing of a threshold halfway between the trace's median and its 99.9th
# percentile (helper). On correct trials a lick follows the cue within the 1.4 s the trace covers.
lk = []
for s in sessions:
    g = s["good"] & np.isfinite(s["cue"]) & (s["trial_types"]["HitR"] | s["trial_types"]["HitL"]) & ~s["trial_types"]["LickEarly"]
    if g.sum() == 0:
        continue
    rt = (s["first_lick"] - s["cue"])[g]
    lk.append({"session": s["name"], "experiment": EXP[s["name"]], "trace_range": s["lick_p999"] - s["lick_median"],
               "median_first_lick_s": np.nanmedian(rt) if np.isfinite(rt).any() else np.nan, "frac_first_lick_under_20ms": np.nanmean(rt < 0.02),
               "trace_seconds_after_cue": np.nanmedian((s["trace_end"] - s["cue"])[g])})
lk = pd.DataFrame(lk)
usable = (lk.median_first_lick_s > 0.05) & (lk.frac_first_lick_under_20ms < 0.2)
print("Sessions with correct trials:", len(lk), "; sessions whose first threshold crossing after the cue looks like a lick (median > 50 ms, < 20% within 20 ms):", int(usable.sum()))
print(lk.groupby(usable.map({True: "lick-like", False: "noise-like"})).agg(sessions=("session", "size"), trace_range=("trace_range", "median"),
      median_first_lick_s=("median_first_lick_s", "median")).round(4).to_string())
print("Trace length after the cue (s), median over sessions:", round(lk.trace_seconds_after_cue.median(), 3))

In [ ]:
# Per-unit measures used by the selectivity and rate claims. The screening trials that set each
# neuron's preferred trial type are drawn with a generator seeded from the unit's position.
t_m = time.time()
M = []
for k, u in enumerate(units):
    e = EXP[u["session"]]
    M.append(alm3.unit_measures(u, SD[u["session"]], N_SCREEN[e], CONDS[e], seed=1000 + k))
info["nR"] = [m["nR"] for m in M]
info["nL"] = [m["nL"] for m in M]
info["fr_mean"] = [m["fr_mean"] for m in M]
print(f"measures for {len(M)} units in {time.time() - t_m:.0f} s")

## units-and-sessions

Claim: "1012 single units were recorded in left ALM during behaviour across 58 sessions (20
unilateral, 38 bilateral) from 12 mice; 72 are fast-spiking, 890 putative pyramidal and 50 of
intermediate width are excluded." Criterion: counts over the non-callosotomy sessions, units
classed by the spike-width bounds of the Methods. The data set gives each unit a `cellType`
label; the trough-to-peak width of the mean of the 29-sample waveforms (19.53 kHz, so widths
come in steps of 0.051 ms) is computed as a second reading, and a third reading upsamples the
mean waveform 10 times by cubic spline before measuring.

In [ ]:
from scipy.interpolate import CubicSpline

def width_upsampled(mw, factor=10):
    x = np.arange(len(mw))
    xs = np.linspace(0, len(mw) - 1, (len(mw) - 1) * factor + 1)
    y = CubicSpline(x, mw)(xs)
    tr = int(np.argmin(y))
    pk = tr + int(np.argmax(y[tr:]))
    return (xs[pk] - xs[tr]) / alm3.SAMPLE_RATE_WAVEFORM * 1e3

info["width_up_ms"] = [width_upsampled(SD[u["session"]]["units"][u["unit"]]["mean_waveform"].astype(float)) for u in units]
def width_class(w):
    return np.where(w > 0.45, "pyramidal", np.where(w < 0.35, "FS", "intermediate"))
info["class_width"] = width_class(info.spike_width_ms)
info["class_width_up"] = width_class(info.width_up_ms)
nc = info[info.experiment != "ccx"]
print("Non-callosotomy sessions:", nc.session.nunique(), "with units;", (ses.experiment != "ccx").sum(), "in all;",
      "unilateral", (ses.experiment == "unilateral").sum(), "; bilateral (exp2, exp2b, exp3)",
      ses.experiment.isin(["exp2", "exp2b", "exp3"]).sum(), "; mice", nc.mouse.nunique())
print("Single units:", len(nc))
print("\nBy the data set's cellType label:", nc.cell_type.value_counts().to_dict())
print("By the width of the mean waveform:", nc.class_width.value_counts().to_dict())
print("By the width of the upsampled mean waveform:", nc.class_width_up.value_counts().to_dict())
print("\nLabel against width class (mean waveform):")
print(pd.crosstab(nc.cell_type, nc.class_width).to_string())
print("\nLabel against width class (upsampled):")
print(pd.crosstab(nc.cell_type, nc.class_width_up).to_string())
print("\nWidth (ms) by label, mean waveform:")
print(nc.groupby("cell_type").spike_width_ms.describe().round(3).to_string())
print("\nPaper: 1012 units, 20 + 38 sessions, 12 mice, 890 pyramidal, 72 FS, 50 intermediate.")
print("Units per experiment:", nc.experiment.value_counts().to_dict())

**Result: not reproduced.** The non-callosotomy sessions hold 1012 single units in 58 sessions (20 unilateral, 38 bilateral) from 12 mice, and 890 are labelled pyramidal, all as the paper reports. The FS count differs: 89 units are labelled FS and 33 carry no label, against 72 and 50 in the paper; by the width of the deposited mean waveforms there are 98 FS and 24 intermediate units, and 104 and 19 after upsampling. Five of the six counts match exactly; the criterion needs all of them, or the class counts within 10%, and the FS count is 24% above the paper's.

## ed1-selective-fraction

Claim: "Most ALM pyramidal neurons distinguish trial types (634 of 890), 176 during the sample,
337 during the delay and 493 during the response epoch." Two-tailed t-test of epoch spike rates,
lick right against lick left control trials, p < 0.05. Readings: trials (a) correct only,
(b) correct and error grouped by instructed movement (the code's choice, `i_yes_trial` in
analysis_compile_all_data_unilateral.m), each with lick-early trials excluded or included (M1);
response epoch the 1.3 s after the cue (code), the first 400 ms, or 1.5 s (M2).

In [ ]:
pyr = (info.cell_type == "pyramidal") & (info.experiment != "ccx")
rows = []
for trials_name, label in (("correct", "correct, lick-early excluded"), ("all", "correct+error, lick-early excluded"), ("all_le", "correct+error, lick-early included")):
    P = np.array([M[i][f"p_sel_{trials_name}"] for i in np.flatnonzero(pyr)])
    for j, resp in ((2, "1.3 s"), (3, "0.4 s"), (4, "1.5 s")):
        sig = P[:, [0, 1, j]] < 0.05
        rows.append({"trials": label, "response epoch": resp, "any": int(sig.any(1).sum()), "sample": int(sig[:, 0].sum()),
                     "delay": int(sig[:, 1].sum()), "response": int(sig[:, 2].sum()), "n": int(len(P)),
                     "untestable": int(np.isnan(P[:, 0]).sum())})
ed1 = pd.DataFrame(rows)
exp_ = {"any": 634, "sample": 176, "delay": 337, "response": 493}
for k, v in exp_.items():
    ed1[f"{k}_ok"] = (ed1[k] - v).abs() <= 0.1 * v
print(ed1.to_string(index=False))
print("Paper: 634 any, 176 sample, 337 delay, 493 response of 890; tolerance 10%.")
print("Readings with all four counts within tolerance:", int(ed1[[f"{k}_ok" for k in exp_]].all(1).sum()))
print("\nBy mouse (code reading: correct+error, lick-early excluded, 1.3 s):")
P = np.array([M[i]["p_sel_all"] for i in np.flatnonzero(pyr)])
bym = pd.DataFrame({"mouse": info[pyr].mouse.values, "any": (P[:, [0, 1, 2]] < 0.05).any(1)}).groupby("mouse")["any"].agg(["sum", "size", "mean"])
print(bym.round(2).to_string())

**Result: reproduced.** With correct control trials, lick-early trials excluded and the 1.3 s response epoch, 633 of 890 pyramidal neurons are selective in some epoch, 176 in the sample, 336 in the delay and 493 in the response epoch, against 634, 176, 337 and 493. The 1.5 s response epoch also matches (635, 490 in the response epoch). With correct and error trials, the reading of the deposited code, the counts are lower (571, 152, 287, 404).

In [ ]:
# Shared pieces for the rate and selectivity recovery claims.
GT = alm3.GRID_T
# Photoinhibition offsets from the cue (end of the ramp), as the code uses them and as the laser
# trace shows (data check above): unilateral sample -1.6 s, early delay -0.8 s, middle delay
# -0.3 s; all 0.8 s conditions -0.9 s.
OFFSET = {1: -1.6, 2: -0.8, 3: -0.3, 4: -0.9, 5: -0.9, 6: -0.9, 7: -0.9, 8: -0.9, 9: -0.9, 10: -0.9, 11: -0.9, 12: -0.9}
ONSET = {1: -2.1, 2: -1.3, 3: -0.8, 4: -1.7, 5: -1.7, 6: -1.7, 7: -1.7, 8: -1.7, 9: -1.7, 10: -1.7, 11: -1.7, 12: -1.7}
RAMP = {1: 0.1, 2: 0.1, 3: 0.1}

# Readings of M4 for the screening trials that set each neuron's preferred type: "code", the
# deposited scripts' rule (5 trials of each type in Experiment 1, 10 elsewhere); "10 per type",
# M4 reading (a); "10 in total", M4 reading (b). Measures other than the code's 200 ms traces are
# computed when first needed, with the same seed per unit.
MV_CACHE = {}

def MV(k, screen="code", width=0.2):
    if screen == "code" and width == 0.2:
        return M[k]
    key = (k, screen, width)
    if key not in MV_CACHE:
        u = units[k]
        e = EXP[u["session"]]
        n, mode = {"code": (N_SCREEN[e], "per_type"), "10 per type": (10, "per_type"), "10 in total": (10, "total")}[screen]
        MV_CACHE[key] = alm3.unit_measures(u, SD[u["session"]], n, CONDS[e], seed=1000 + k, width_psth=width, screen=mode)
    return MV_CACHE[key]

def screens(exp):
    return ("code", "10 per type", "10 in total") if N_SCREEN[exp] != 10 else ("code", "10 in total")

def selective(k, trials="all", rule="code"):
    """Selective by the t-test of M3. rule "code": in the sample or delay epoch (the deposited
    scripts, Fig. 1g and 2e sections); rule "M3": in any of the sample, delay or response (1.3 s)
    epochs (claims file, M3)."""
    p = np.nan_to_num(M[k][f"p_sel_{trials}"][:3], nan=1)
    return bool((p[:2] < 0.05).any()) if rule == "code" else bool((p < 0.05).any())

def selection(exp, conds, sel_trials="all", min_stim=2, min_ctrl=5, fr_min=None, pref_min=1.0, experiments=None,
              rule="code", screen="code", steps=False):
    """Indices of the neurons of the selectivity recovery figures: putative pyramidal, enough
    control trials for the screening subset, selective (rule), more than `min_ctrl` control and
    more than `min_stim` trials of each type in every condition. Rule "code" adds the scripts'
    |preference count| > 1 (Fig. 1g and 2e sections); rule "M3" has no such filter. With steps,
    also returns the number of neurons left after each step."""
    exps = experiments or [exp]
    out = []
    n = {"pyramidal": 0, "screening trials": 0, "selective": 0, "|preference| > 1": 0, "control trials": 0, "trials per condition": 0}
    for k, u in enumerate(units):
        if EXP[u["session"]] not in exps or u["cell_type"] != "pyramidal":
            continue
        n["pyramidal"] += 1
        m = MV(k, screen)
        if m["pref_counts"] is None:
            continue
        n["screening trials"] += 1
        if not selective(k, sel_trials, rule):
            continue
        n["selective"] += 1
        if rule == "code" and abs(m["pref_counts"][0]) <= pref_min:
            continue
        n["|preference| > 1"] += 1
        if m["nR"] <= min_ctrl or m["nL"] <= min_ctrl:
            continue
        n["control trials"] += 1
        if any(c not in m["cond"] or m["cond"][c]["nR"] <= min_stim or m["cond"][c]["nL"] <= min_stim for c in conds):
            continue
        n["trials per condition"] += 1
        if fr_min is not None and not (m["fr_mean"] > fr_min):
            continue
        out.append(k)
    return (out, n) if steps else out

def sel_traces(ks, cond, width=0.2, screen="code", pref="whole"):
    """Selectivity (preferred minus non-preferred, spikes/s) on control test trials and on
    photoinhibition trials of `cond`, neurons x time. pref "whole": preferred type from the count
    from sample onset to 1.3 s after the cue (M4 reading a; code); "best": from the epoch where the
    neuron is most selective on the screening trials (M4 reading b)."""
    ctrl, stim = [], []
    for k in ks:
        m0 = MV(k, screen)
        m = MV(k, screen, width)
        sgn = (1 if m0["pref_counts"][0] >= 0 else -1) if pref == "whole" else m0["pref_best"][1]
        ctrl.append(sgn * (m["psth_ctrl"][1] - m["psth_ctrl"][-1]))
        c = m["cond"][cond]["psth"]
        stim.append(sgn * (c[1] - c[-1]))
    return np.array(ctrl), np.array(stim)

In [ ]:
def recovery_table(exp, conds, t_end, sel_trials, min_stim, width, stamp_shift, seed, experiments=None, min_ctrl=5,
                   rule="code", screen="code", pref="whole"):
    ks = selection(exp, conds, sel_trials=sel_trials, min_stim=min_stim, min_ctrl=min_ctrl, experiments=experiments, rule=rule, screen=screen)
    rng = np.random.default_rng(seed)
    rows = []
    for c in conds:
        ctrl, stim = sel_traces(ks, c, width=width, screen=screen, pref=pref)
        mean, sd, frac, cens = alm3.recovery_time(ctrl, stim, GT + stamp_shift, OFFSET[c], t_end, rng)
        ratio = (ctrl - stim).mean(0) / ctrl.mean(0)
        late = float(ratio[np.argmin(np.abs(GT + stamp_shift - (-0.2)))])
        rows.append({"condition": c, "label": alm3.STIM_LABELS[c], "neurons": len(ks), "mice": len({units[k]["mouse"] for k in ks}),
                     "recovery_ms": mean * 1000, "sd_ms": sd * 1000, "frac_boot_recovered": frac, "censored_ms": cens * 1000,
                     "deficit_at_-0.2s": late})
    return pd.DataFrame(rows), ks

def mean_rate_traces(exp, conds, fr_min=1.0, min_ctrl=8, min_stim=3, experiments=None):
    """Population mean rate (spikes/s, 200 ms boxcar) on control and photoinhibition trials, lick
    right and lick left averaged per neuron; neurons as in the code's Fig. 1f and 2d sections."""
    exps = experiments or [exp]
    out = {}
    for c in conds:
        ks = [k for k, u in enumerate(units) if EXP[u["session"]] in exps and u["cell_type"] == "pyramidal"
              and M[k]["nR"] > min_ctrl and M[k]["nL"] > min_ctrl and c in M[k]["cond"]
              and M[k]["cond"][c]["nR"] > min_stim and M[k]["cond"][c]["nL"] > min_stim and M[k]["fr_mean"] > fr_min]
        ctrl = np.array([(M[k]["psth_ctrl_all"][1] + M[k]["psth_ctrl_all"][-1]) / 2 for k in ks])
        stim = np.array([(M[k]["cond"][c]["psth"][1] + M[k]["cond"][c]["psth"][-1]) / 2 for k in ks])
        P = np.array([M[k]["cond"][c]["p_rate"] for k in ks])
        out[c] = {"ks": ks, "ctrl": ctrl, "stim": stim, "p": P}
    return out

def rate_summary(R, times):
    rows = []
    for c, d in R.items():
        r = {"condition": c, "label": alm3.STIM_LABELS[c], "neurons": len(d["ks"])}
        for t in times:
            i = np.argmin(np.abs(GT - t))
            j = np.argmin(np.abs(alm3.RATE_T - t))
            r[f"rate_ratio@{t:+.2f}"] = d["stim"][:, i].mean() / d["ctrl"][:, i].mean()
            r[f"frac_changed@{t:+.2f}"] = (d["p"][:, j] < 0.01).mean()
        rows.append(r)
    return pd.DataFrame(rows)

WINDOWS = ((0.2, 0.0, "200 ms, centre"), (0.4, 0.2, "400 ms, end"))

def recovery_readings(exp, conds, t_end, expected, seed, experiments=None):
    """Recovery times under every reading of M4 and of the selection: trials of the selectivity
    test, minimum trials per condition, window and time stamp, screening subset, preference window
    and selection rule. A condition is within tolerance only when most resamples (at least half)
    reach 80% before `t_end` and their mean recovery time is within tolerance of the paper's;
    'censored_ms' counts the resamples that do not recover at `t_end`."""
    rows = []
    for rule in ("code", "M3"):
        for screen in screens(exp):
            for pref in ("whole", "best"):
                for sel_trials in ("all", "correct"):
                    for min_stim in (2, 3):
                        for width, shift, lab in WINDOWS:
                            tab, ks = recovery_table(exp, conds, t_end, sel_trials, min_stim, width, shift, seed=seed, experiments=experiments,
                                                     rule=rule, screen=screen, pref=pref)
                            for r in tab.to_dict("records"):
                                rows.append({"rule": rule, "screen": screen, "pref": pref, "selective_on": sel_trials, "min_trials": f">{min_stim}", "window": lab, **r})
    t = pd.DataFrame(rows)
    t["within_tol"] = [(r.frac_boot_recovered >= 0.5 and abs(r.recovery_ms - expected[r.condition][0]) <= expected[r.condition][1]) if r.condition in expected else np.nan
                       for r in t.itertuples()]
    return t

READING = ["rule", "screen", "pref", "selective_on", "min_trials", "window"]

def readings_wide(t):
    """One row per reading: neurons, and per condition the conditional mean recovery time, the
    fraction of resamples that recover, the censored mean and the deficit 0.2 s before the cue."""
    w = t.pivot_table(index=READING, columns="condition", values=["recovery_ms", "frac_boot_recovered", "censored_ms", "deficit_at_-0.2s"], aggfunc="first")
    w.columns = [f"{a.replace('frac_boot_recovered', 'frac').replace('recovery_ms', 'ms').replace('censored_ms', 'cens_ms').replace('deficit_at_-0.2s', 'def')}_{c}" for a, c in w.columns]
    w = w[sorted(w.columns, key=lambda x: (x.split("_")[-1], x))]
    w.insert(0, "neurons", t.groupby(READING).neurons.first())
    return w.reset_index()

## fig1f-rate-recovery

Claim: "Within 400 ms after unilateral photoinhibition, spike rates are indistinguishable from
control in 90% of neurons; only 10% keep a sustained change." Per neuron and condition of
Experiment 1, the rate test (400 ms windows every 50 ms, control against photoinhibition, lick
right and lick left pooled, p < 0.01). Neurons: putative pyramidal, mean rate > 1 spike/s, more
than 8 control trials of each type and more than 3 photoinhibition trials of each type (code,
Fig. 1f section: `i_cell`, `i_n_trial`); without the rate exclusion as a reading (M3). "Within
400 ms after" read as (a) the 400 ms window starting at offset (centre offset + 0.2 s) and (b)
the window centred 400 ms after offset. The code's own measure, the fraction in the window
centred 0.25 s before the cue (`frac_persist`), is shown too.

In [ ]:
def rate_fraction_table(exp, conds, fr_min=1.0, min_ctrl=8, min_stim=3):
    rows = []
    for c in conds:
        ks = [k for k, u in enumerate(units) if EXP[u["session"]] == exp and u["cell_type"] == "pyramidal"
              and M[k]["nR"] > min_ctrl and M[k]["nL"] > min_ctrl and c in M[k]["cond"]
              and M[k]["cond"][c]["nR"] > min_stim and M[k]["cond"][c]["nL"] > min_stim
              and M[k]["cond"][c]["p_rate"] is not None and (fr_min is None or M[k]["fr_mean"] > fr_min)]
        P = np.array([M[k]["cond"][c]["p_rate"] for k in ks])
        frac = (P < 0.01).mean(0)
        def at(t):
            return float(frac[np.argmin(np.abs(alm3.RATE_T - t))])
        rows.append({"condition": c, "label": alm3.STIM_LABELS[c], "neurons": len(ks), "mice": len({units[k]["mouse"] for k in ks}),
                     "during": at(OFFSET[c] - 0.25), "a_window_from_offset": at(OFFSET[c] + 0.2),
                     "b_centre_offset+0.4": at(OFFSET[c] + 0.4), "code_frac_persist(-0.25)": at(-0.25),
                     "before_onset": at(ONSET[c] - 0.25)})
    return pd.DataFrame(rows), ks

for fr_min in (1.0, None):
    tab, _ = rate_fraction_table("unilateral", [1, 2, 3], fr_min=fr_min)
    print(f"Rate exclusion: {'mean rate > 1 spike/s' if fr_min else 'none'}")
    print(tab.round(3).to_string(index=False))
    for col in ("a_window_from_offset", "b_centre_offset+0.4"):
        v = tab[col].values
        print(f"  {col}: per condition within 0.1 +- 0.01: {list(np.abs(v - 0.1) <= 0.01)}; mean {v.mean():.3f}; all below 0.2: {bool((v < 0.2).all())}")
    print()
print("Paper: 168, 168, 175 neurons; 10% with a sustained change. 'during' is the window centred 0.25 s before offset; 'before_onset' the window centred 0.25 s before onset (the false-positive level of the test).")

**Result: partially reproduced.** The code's selection gives exactly the paper's 168, 168 and 175 neurons. In the window centred 400 ms after offset (reading b), 13.1%, 14.3% and 9.1% of neurons still have a significant rate change (mean 12.2%), and in the window starting at offset (reading a) 17.9%, 19.6% and 10.9%; only the third condition is within 10 +- 1%, and all are below 20%, the criterion's bound for partial reproduction. Before photoinhibition the test flags 0 to 1.1% of neurons.

## fig1g-selectivity-recovery

Claim: recovery of > 80% of control selectivity within 514 ms; recovery times 373 +- 260 ms
(sample), 510 +- 218 ms (early delay) and 327 +- 112 ms (window ending 0.3 s before the cue);
55 selective neurons. Method of the code (Fig. 1g section of analysis_compile_all_data_unilateral.m),
which fills the open details of M4: preference from 5 control trials of each type drawn at
random (10 in total, the Methods' "subset of 10 trials"), on the spike count from sample onset to
1.3 s after the cue; selectivity traces on the remaining trials, 200 ms boxcar centred on each
time; recovery time the mean over 1000 bootstrap resamples of neurons of the first time after
offset (and before 1 s after the cue) at which (control minus photoinhibition) / control mean
selectivity < 0.2, +- the standard deviation over resamples. Readings shown: trials of the
selectivity test (all, the code's, or correct, M3), more than 2 (code) or more than 3 (legend)
trials per condition, 200 ms windows stamped at their centre or 400 ms windows stamped at their
end (M4).

Every reading of M4 and of the selection is run, as the criterion asks (verifier, round 1):
the selection rule, "code" (selective in the sample or delay epoch with |preference count| > 1,
the scripts) or "M3" (selective in any of the sample, delay or response epochs, the claims file);
the screening trials, the code's, 10 of each type (M4 reading a) or 10 in total (M4 reading b);
the preference, from the whole count (M4 reading a) or from the epoch where the screening trials
differ most (M4 reading b); the trials of the selectivity test; the minimum trials per condition;
and the window. A condition counts as within tolerance only when at least half the resamples
reach 80% before the end of the search and their mean is within tolerance; the mean with the
other resamples counted at the end of the search is also shown.

In [ ]:
f1g = recovery_readings("unilateral", [1, 2, 3], 1.0, {1: (373, 260), 2: (510, 218), 3: (327, 112)}, seed=11)
w1g = readings_wide(f1g)
ok = f1g.groupby(READING).agg(all_within=("within_tol", "all"), all_recover=("frac_boot_recovered", lambda x: bool((x >= 0.5).all())))
w1g = w1g.merge(ok.reset_index(), on=READING)
w1g["neurons_within"] = (w1g.neurons - 55).abs() <= 5.5
print("Per reading: neurons; per condition (1 sample, 2 early delay, 3 middle delay) the mean recovery time over resamples that recover (ms_c),")
print("the fraction of resamples that recover before 1 s after the cue (frac_c), the mean with the others counted at 1 s after the cue (cens_ms_c)")
print("and (control - photoinhibition) / control 0.2 s before the cue (def_c):")
print(w1g.round(2).to_string(index=False))
print(f"\nReadings: {len(w1g)}; all three times within tolerance: {int(w1g.all_within.sum())}; every condition recovers in at least half the resamples: "
      f"{int(w1g.all_recover.sum())}; neuron count within 55 +- 5.5: {int(w1g.neurons_within.sum())} (range {w1g.neurons.min()} to {w1g.neurons.max()})")
code_row = w1g[(w1g["rule"] == "code") & (w1g.screen == "code") & (w1g.pref == "whole") & (w1g.selective_on == "all")
               & (w1g.min_trials == ">2") & (w1g.window == "200 ms, centre")].iloc[0]
print("Fraction of resamples recovering, code reading:", ", ".join(f"{code_row[f'frac_{c}']:.3f}" for c in (1, 2, 3)),
      "; lowest over all readings:", ", ".join(f"{w1g[f'frac_{c}'].min():.3f}" for c in (1, 2, 3)))
print("\nNeurons left after each selection step (selective on all trials, > 2 trials per condition):")
print(pd.DataFrame([{"rule": r, "screen": sc, **selection("unilateral", [1, 2, 3], rule=r, screen=sc, steps=True)[1]}
                    for r in ("code", "M3") for sc in screens("unilateral")]).to_string(index=False))
print("Paper: 55 neurons; 373 +- 260, 510 +- 218, 327 +- 112 ms. Rule 'code': selective in the sample or delay epoch with |preference| > 1 (scripts);")
print("'M3': selective in any epoch (claims file). Screen: trials that set the preference. Pref: 'whole' count, or 'best' epoch.")

**Result: partially reproduced.** Under the code's reading (selective on all trials, more than 2 trials per condition, 200 ms windows), selectivity recovers to 80% of control 347 ms after the sample photoinhibition, 420 ms after the early delay one and 236 ms after the middle delay one, within the paper's 373 +- 260, 510 +- 218 and 327 +- 112 ms, and at least 99% of resamples recover in each condition. In all 96 readings every condition recovers in at least 96% of resamples, and 34 put all three times within tolerance. No reading gives the paper's 55 neurons: the selections hold 66 to 160 neurons (72 under the code's reading), so the times match on a different neuron set. Of the 292 Experiment 1 pyramidal neurons, 99 are selective in the sample or delay epoch and 72 remain after the code's preference and trial filters; selection by any epoch (M3) leaves 135 to 137.

## fig2-bilateral-recovery

Claim: spike rate and selectivity of left ALM neurons recover within 600 ms after ipsilateral
photoinhibition, contralateral photoinhibition hardly affects activity, and after bilateral
photoinhibition the rate recovers on average but selectivity does not; recovery 538 +- 178 ms
(ipsilateral), 192 +- 114 ms (contralateral), none (bilateral); 143 selective neurons.
Experiment 2 sessions (types 4, 5, 6). Method of the code (analysis_compile_all_data_bilateral.m,
Fig. 2e section): as for fig1g, with 10 screening trials of each type and the recovery searched
from offset (0.9 s before the cue) up to the cue. "Spike rate (> 90 %)" read as (a) more than 90%
of neurons without a significant rate change in the window centred 600 ms after offset, or (b)
population mean rate within 10% of control there. Rate neurons as in the code's Fig. 2d
section (pyramidal, mean rate > 1 spike/s, > 8 control and > 3 photoinhibition trials per type).

Every reading of M4 and of the selection is run, as the criterion asks (verifier, round 1):
the selection rule, "code" (selective in the sample or delay epoch with |preference count| > 1,
the scripts) or "M3" (selective in any of the sample, delay or response epochs, the claims file);
the screening trials, the code's, 10 of each type (M4 reading a) or 10 in total (M4 reading b);
the preference, from the whole count (M4 reading a) or from the epoch where the screening trials
differ most (M4 reading b); the trials of the selectivity test; the minimum trials per condition;
and the window. A condition counts as within tolerance only when at least half the resamples
reach 80% before the end of the search and their mean is within tolerance; the mean with the
other resamples counted at the end of the search is also shown.

In [ ]:
for fr_min, lab in ((1.0, "neurons above 1 spike/s (code)"), (0.0, "no rate exclusion")):
    R2 = mean_rate_traces("exp2", [4, 5, 6], fr_min=fr_min)
    rs2 = rate_summary(R2, [-1.3, -0.3, -0.05])
    print(f"Rates, {lab}: ratio of photoinhibition to control population mean rate and fraction of neurons with a significant change (p < 0.01),")
    print("at 0.4 s after onset (-1.3), 0.6 s after offset (-0.3) and just before the cue (-0.05):")
    print(rs2.drop(columns="label").round(3).to_string(index=False))
f2 = recovery_readings("exp2", [4, 5, 6], 0.0, {4: (538, 178), 5: (192, 114)}, seed=21)
w2 = readings_wide(f2)
ok = f2.groupby(READING).agg(times_within=("within_tol", lambda x: bool(x.dropna().all())))
w2 = w2.merge(ok.reset_index(), on=READING)
w2["bilateral_no_recovery"] = w2.frac_6 < 0.5
w2["neurons_within"] = (w2.neurons - 143).abs() <= 14.3
print("\nSelectivity recovery, every reading (4 ipsilateral, 5 contralateral, 6 bilateral; columns as for fig1g-selectivity-recovery, censored at the cue):")
print(w2.round(2).to_string(index=False))
print(f"\nReadings: {len(w2)}; ipsilateral and contralateral times within tolerance (each from at least half the resamples): {int(w2.times_within.sum())}; "
      f"of these with bilateral not recovering: {int((w2.times_within & w2.bilateral_no_recovery).sum())}, and with the neuron count within 143 +- 14.3: "
      f"{int((w2.times_within & w2.bilateral_no_recovery & w2.neurons_within).sum())}")
print("Ipsilateral: fraction of resamples recovering", round(w2.frac_4.min(), 2), "to", round(w2.frac_4.max(), 2), "; deficit 0.2 s before the cue",
      round(w2.def_4.min(), 2), "to", round(w2.def_4.max(), 2))
six = w2[w2.times_within & w2.bilateral_no_recovery]
print(f"In the {len(six)} readings with both times within tolerance and bilateral not recovering: ipsilateral deficit 0.2 s before the cue",
      round(six.def_4.min(), 2), "to", round(six.def_4.max(), 2), "; fraction of resamples recovering", round(six.frac_4.min(), 2), "to", round(six.frac_4.max(), 2))
print("\nNeurons left after each selection step:")
print(pd.DataFrame([{"rule": r, "screen": sc, **selection("exp2", [4, 5, 6], rule=r, screen=sc, steps=True)[1]}
                    for r in ("code", "M3") for sc in screens("exp2")]).to_string(index=False))
ks2 = selection("exp2", [4, 5, 6])
ctrl2, ipsi2 = sel_traces(ks2, 4)
pts = [-1.7, -1.3, -0.9, -0.7, -0.5, -0.3, -0.1]
print(f"\nCode reading ({len(ks2)} neurons): mean selectivity (spikes/s, 200 ms windows) at times from the cue")
print(pd.DataFrame({"t_s": pts, "control": [ctrl2[:, np.argmin(np.abs(GT - t))].mean() for t in pts],
                    "ipsilateral": [ipsi2[:, np.argmin(np.abs(GT - t))].mean() for t in pts]}).round(2).to_string(index=False))
print("\nPaper: 143 neurons; ipsilateral 538 +- 178 ms, contralateral 192 +- 114 ms, bilateral no recovery.")

**Result: partially reproduced.** After ipsilateral photoinhibition the population rate is back to 0.96 of control 600 ms after offset (0.96 without the rate exclusion) and 9.4% of neurons have a significant change there, so the rate recovers under both readings. Contralateral photoinhibition leaves the rate at 0.90 of control during photoinhibition, and its selectivity recovers in 240 ms (code reading), within 192 +- 114 ms. Bilateral selectivity reaches 80% before the cue in at most 1% of resamples in every reading, but the bilateral rate is 1.11 of control just before the cue (1.13 without the rate exclusion), outside the criterion's 10%. Ipsilateral selectivity recovers in some readings only: under the code's reading 25% of resamples reach 80% before the cue (mean 716 ms; 854 ms counting the others at the cue), and across the 64 readings 0% to 79% do. Six readings, all with the 10 screening trials drawn in total (M4 reading b), have at least half the resamples recovering with both times within tolerance and bilateral selectivity not recovering; one of them (code rule, preference from the most selective epoch) also has 146 neurons, within 143 +- 14.3 (684 ms from 51% of resamples, contralateral 229 ms). In every reading the ipsilateral selectivity 0.2 s before the cue is still 28% to 60% below control, and 28% to 43% in those six readings, so where it reaches 80% it falls back before the cue. The ordering (ipsilateral recovers, bilateral does not) holds under those six readings, and the bilateral rate misses the criterion, so the claim holds in part, on few readings. The criterion's partial rule names a time outside tolerance; here both times are within tolerance and the bilateral rate fails, so the partial outcome follows the protocol's general meaning (the effect holds only in part), not the criterion's wording.

## ed7-unilateral-4-spots-bilateral-1-spot

Claim: bilateral photoinhibition with one spot per side causes a larger persistent change in
selectivity than unilateral photoinhibition with four spots; recovery 490 +- 280 ms (ipsilateral
4 spots), 235 +- 156 ms (contralateral 4 spots), none (bilateral 1 spot); 40 selective neurons.
Experiment 3 sessions (types 7, 8, 9), method as fig2-bilateral-recovery. The "persistent
change" is the mean delta selectivity (photoinhibition minus control, spikes/s) in the last
400 ms of the delay (criterion).

Every reading of M4 and of the selection is run, as the criterion asks (verifier, round 1):
the selection rule, "code" (selective in the sample or delay epoch with |preference count| > 1,
the scripts) or "M3" (selective in any of the sample, delay or response epochs, the claims file);
the screening trials, the code's, 10 of each type (M4 reading a) or 10 in total (M4 reading b);
the preference, from the whole count (M4 reading a) or from the epoch where the screening trials
differ most (M4 reading b); the trials of the selectivity test; the minimum trials per condition;
and the window. A condition counts as within tolerance only when at least half the resamples
reach 80% before the end of the search and their mean is within tolerance; the mean with the
other resamples counted at the end of the search is also shown. As a variant for a question to a reader, the bilateral 1-spot neurons of the sessions
with types 4, 5 and 7 are pooled with Experiment 3 (ED Fig. 2b gives n = 102 for that condition).

In [ ]:
e7 = recovery_readings("exp3", [7, 8, 9], 0.0, {8: (490, 280), 9: (235, 156)}, seed=31)
w7 = readings_wide(e7)
ok = e7.groupby(READING).agg(times_within=("within_tol", lambda x: bool(x.dropna().all())))
w7 = w7.merge(ok.reset_index(), on=READING)
w7["bilateral_no_recovery"] = w7.frac_7 < 0.5
print("Every reading (7 bilateral 1 spot, 8 ipsilateral 4 spots, 9 contralateral 4 spots):")
print(w7.round(2).to_string(index=False))
print(f"\nReadings: {len(w7)}; times within tolerance: {int(w7.times_within.sum())}; bilateral 1 spot reaching 80% before the cue in fewer than half the resamples: "
      f"{int(w7.bilateral_no_recovery.sum())}; both: {int((w7.times_within & w7.bilateral_no_recovery).sum())}")
print("Bilateral 1 spot: fraction of resamples recovering", round(w7.frac_7.min(), 2), "to", round(w7.frac_7.max(), 2))
print("\nNeurons left after each selection step:")
print(pd.DataFrame([{"rule": r, "screen": sc, **selection("exp3", [7, 8, 9], rule=r, screen=sc, steps=True)[1]}
                    for r in ("code", "M3") for sc in screens("exp3")]).to_string(index=False))
print("\nMean delta selectivity (photoinhibition minus control, spikes/s) in the last 400 ms of the delay:")
win = (GT > -0.4) & (GT <= 0)
for rule in ("code", "M3"):
    ks7 = selection("exp3", [7, 8, 9], sel_trials="all", min_stim=2, rule=rule)
    ctrl, s7 = sel_traces(ks7, 7)
    _, s8 = sel_traces(ks7, 8)
    d7, d8 = (s7 - ctrl)[:, win].mean(1), (s8 - ctrl)[:, win].mean(1)
    print(f"  rule {rule}, {len(ks7)} neurons: bilateral 1 spot {d7.mean():+.2f} (s.e.m. {d7.std(ddof=1) / np.sqrt(len(d7)):.2f}), ipsilateral 4 spots {d8.mean():+.2f} "
          f"(s.e.m. {d8.std(ddof=1) / np.sqrt(len(d8)):.2f}); control selectivity {ctrl[:, win].mean():.2f}; "
          f"difference {np.mean(d7 - d8):+.2f}, paired t-test p = {stats.ttest_rel(d7, d8).pvalue:.3g}")
# Variant for the question to a reader: ED Fig. 2b gives Experiment 3 n = 52, 52, 102, which suggests that the bilateral
# 1-spot neurons of the sessions with types 4, 5, 7 were pooled with Experiment 3 for that condition.
print("\nVariant: bilateral 1 spot (type 7) with the sessions holding types 4, 5, 7 pooled with Experiment 3:")
rows = []
for rule in ("code", "M3"):
    for sel_trials in ("all", "correct"):
        for width, shift, lab in WINDOWS:
            tab, ks = recovery_table("exp3", [7], 0.0, sel_trials, 2, width, shift, seed=32, experiments=["exp3", "exp2b"], rule=rule)
            rows.append({"rule": rule, "selective_on": sel_trials, "window": lab, **tab.iloc[0].drop(["label", "condition"]).to_dict()})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nPaper: 40 neurons; 490 +- 280 and 235 +- 156 ms; bilateral 1 spot does not recover.")

**Result: not reproduced.** In the six Experiment 3 sessions, the ipsilateral and contralateral 4-spot recovery times are within tolerance in 40 of 64 readings (code reading 530 ms and 119 ms, against 490 +- 280 and 235 +- 156 ms). But bilateral 1-spot selectivity reaches 80% of control before the cue in 36% to 99% of resamples (80% under the code's reading), where the paper reports no recovery; in no reading are both the times within tolerance and the bilateral selectivity unrecovered. The bilateral 1-spot change in the last 400 ms of the delay is larger in magnitude than the ipsilateral 4-spot one (-0.55 against -0.21 spikes/s with 32 neurons, -0.46 against -0.11 with 59), but neither difference is significant (paired p = 0.53 and 0.27). The selections hold 29 to 61 neurons against 40. With the bilateral 1-spot neurons of the sessions with types 4, 5 and 7 pooled (54 to 121 neurons), 32% to 77% of resamples recover.

## ed2b-silencing

Claim: photoinhibition silences more than 90% of putative pyramidal activity, similarly with
ipsilateral and bilateral photoinhibition and in both mouse lines, while contralateral
photoinhibition barely changes the average rate. Per neuron, spike rate during photoinhibition
over the rate in the same window of control trials, neurons with mean rate < 1 spike/s excluded
(criterion). Window (a) photoinhibition before the ramp, (b) the whole period with the ramp.
Reduction = 1 minus the mean normalized rate. Neurons need at least 3 photoinhibition trials and
5 control trials (a standard choice). Mouse lines from the data set's meta data files (gene_modification field), as in the data check.

In [ ]:
def silencing_rows(window):
    rows = []
    for k, u in enumerate(units):
        e = EXP[u["session"]]
        if u["cell_type"] != "pyramidal" or not (M[k]["fr_mean"] >= 1) or e == "none":
            continue
        s = SD[u["session"]]
        tr = u["trials"]
        st = s["stim_type"][tr]
        ok = u["stable"] & ~s["trial_types"]["LickEarly"][tr]
        for c in CONDS[e]:
            a = ONSET[c]
            b = OFFSET[c] - (RAMP.get(c, 0.2) if window == "a" else 0.0)
            ci, si = np.flatnonzero(ok & (st == 0)), np.flatnonzero(ok & (st == c))
            if len(si) < 3 or len(ci) < 5:
                continue
            cnt = alm3.binned_window(u["binned"], [(a + b) / 2], b - a)[:, 0]
            rc, rs = cnt[ci].mean(), cnt[si].mean()
            if rc > 0:
                rows.append({"session": u["session"], "experiment": e, "condition": c, "label": alm3.STIM_LABELS[c], "mouse": u["mouse"],
                             "line": LINE[u["mouse"]], "norm_rate": rs / rc})
    return pd.DataFrame(rows)

for window in ("a", "b"):
    sil = silencing_rows(window)
    print(f"Window ({window}): {'onset to start of ramp' if window == 'a' else 'onset to end of ramp'}")
    t = sil.groupby(["experiment", "condition", "label"]).norm_rate.agg(["size", "mean", "median"])
    t["reduction"] = 1 - t["mean"]
    print(t.round(3).to_string())
    print("By mouse line, ipsilateral and bilateral conditions (types 1-4, 6-8):")
    t2 = sil[sil.condition.isin([1, 2, 3, 4, 6, 7, 8])].groupby(["line", "experiment"]).norm_rate.agg(["size", "mean"])
    t2["reduction"] = 1 - t2["mean"]
    print(t2.round(3).to_string())
    t3 = sil[(sil.condition == 4) & (sil.experiment == "exp2")].groupby("line").norm_rate.agg(["size", "mean"])
    t3["reduction"] = 1 - t3["mean"]
    print("By mouse line, Experiment 2 ipsilateral (type 4), as ED Fig. 2b right:")
    print(t3.round(3).to_string())
    print("By mouse, Experiment 2 ipsilateral (type 4):")
    print(sil[(sil.condition == 4) & (sil.experiment == "exp2")].groupby("mouse").norm_rate.agg(["size", "mean"]).round(3).to_string())
    print()
NOSIL = set(clk[clk.ratio > 0.5].session)
sil = silencing_rows("a")
print("Neurons per condition (window a) against the ED Fig. 2b legend (Experiment 1: 117, 110, 109; Experiment 2: 300, 294, 301; Experiment 3: 52, 52, 102):")
print(sil.groupby(["experiment", "condition"]).size().to_string())
print("Bilateral 1 spot (type 7), Experiment 3 and the sessions with types 4, 5, 7 together:", int((sil.condition == 7).sum()))
print("\nVariant, window (a), without the four sessions in which photoinhibition does not lower the rate (data check):", sorted(NOSIL))
sx = sil[~sil.session.isin(NOSIL)]
t = sx.groupby(["experiment", "condition", "label"]).norm_rate.agg(["size", "mean"])
t["reduction"] = 1 - t["mean"]
print(t.round(3).to_string())
t2 = sx[sx.condition.isin([1, 2, 3, 4, 6, 7, 8])].groupby(["line", "experiment"]).norm_rate.agg(["size", "mean"])
t2["reduction"] = 1 - t2["mean"]
print(t2.round(3).to_string())
t3 = sx[(sx.condition == 4) & (sx.experiment == "exp2")].groupby("line").norm_rate.agg(["size", "mean"])
t3["reduction"] = 1 - t3["mean"]
print("Experiment 2 ipsilateral by line:")
print(t3.round(3).to_string())
print("Criterion: reduction at least 0.81 for ipsilateral and bilateral photoinhibition in each experiment and each line; contralateral mean normalized rate within 0.8 to 1.2.")

**Result: reproduced.** With the window before the offset ramp (reading a), the mean normalized rate of pyramidal neurons gives a reduction of at least 0.91 for ipsilateral and bilateral photoinhibition in Experiment 1 (0.910 to 0.938), Experiment 2 (0.916, 0.924) and Experiment 3 (0.962, 0.940), and 0.885 and 0.886 in the six sessions with types 4, 5 and 7, and in Experiment 2 ipsilateral 0.951 in PV x ReaChR and 0.845 in VGAT-ChR2 mice, all at least 0.81. Contralateral photoinhibition leaves the mean normalized rate at 0.945 (Experiment 2) and 1.152 (Experiment 3), within 0.8 to 1.2. With the whole period (reading b) the Experiment 3 contralateral value is 1.222, outside. Without the four sessions in which photoinhibition does not lower the rate, the VGAT-ChR2 reduction in the sessions with types 4, 5 and 7 rises from 0.750 to 0.975 and in Experiment 2 ipsilateral from 0.845 to 0.852. The neuron counts of Experiments 1 and 2 (180 per condition, 319 to 323) are above the legend's 117, 110, 109 and 300, 294, 301; Experiment 3 has 56 per condition, above the legend's 52 and 52 and below its 102 for bilateral 1 spot.

## ed2e-no-rebound

Claim: no rebound excitation on average after photoinhibition offset; a small proportion of
neurons show rebound, balanced by low-level sustained inhibition in a larger proportion.
Experiment 2, lick right trials, ipsilateral photoinhibition (type 4); windows 0 to 400 ms and
400 to 800 ms after offset; per neuron two-tailed t-test of counts, control against
photoinhibition, p < 0.01; population test a paired t-test over neurons of the mean rates
(criterion). Neurons: pyramidal with more than 3 lick right photoinhibition trials, with and
without the 1 spike/s exclusion (M3).

In [ ]:
for fr_min in (1.0, None):
    rows = []
    for k, u in enumerate(units):
        if EXP[u["session"]] != "exp2" or u["cell_type"] != "pyramidal":
            continue
        if fr_min is not None and not (M[k]["fr_mean"] > fr_min):
            continue
        s = SD[u["session"]]
        tr = u["trials"]
        st, sd_ = s["stim_type"][tr], alm3.side(s)[tr]
        ok = u["stable"] & ~s["trial_types"]["LickEarly"][tr] & (sd_ == 1)
        ci, si = np.flatnonzero(ok & (st == 0)), np.flatnonzero(ok & (st == 4))
        if len(si) <= 3 or len(ci) <= 3:
            continue
        r = {}
        for name, (a, b) in (("w1", (-0.9, -0.5)), ("w2", (-0.5, -0.1))):
            cnt = alm3.binned_window(u["binned"], [(a + b) / 2], b - a)[:, 0]
            r[name + "_ctrl"], r[name + "_stim"] = cnt[ci].mean() / 0.4, cnt[si].mean() / 0.4
            r[name + "_p"] = stats.ttest_ind(cnt[ci], cnt[si]).pvalue
        rows.append(r)
    rb = pd.DataFrame(rows)
    print(f"Rate exclusion: {'> 1 spike/s' if fr_min else 'none'}; neurons {len(rb)}")
    for w, lab in (("w1", "0-400 ms after offset"), ("w2", "400-800 ms after offset")):
        d = rb[w + "_stim"] - rb[w + "_ctrl"]
        t = stats.ttest_rel(rb[w + "_stim"], rb[w + "_ctrl"])
        exc = int(((rb[w + "_p"] < 0.01) & (d > 0)).sum())
        sup = int(((rb[w + "_p"] < 0.01) & (d < 0)).sum())
        print(f"  {lab}: mean control {rb[w + '_ctrl'].mean():.2f}, photoinhibition {rb[w + '_stim'].mean():.2f} spikes/s; "
              f"difference {d.mean():+.2f}, paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}; excited {exc}, suppressed {sup} (p < 0.01)")
        exc5, sup5 = int(((rb[w + "_p"] < 0.05) & (d > 0)).sum()), int(((rb[w + "_p"] < 0.05) & (d < 0)).sum())
        print(f"    at p < 0.05: excited {exc5}, suppressed {sup5}; any difference: higher {int((d > 0).sum())}, lower {int((d < 0).sum())} "
              f"(median difference {d.median():+.2f} spikes/s)")

**Result: not reproduced.** On lick right trials after ipsilateral photoinhibition (306 neurons above 1 spike/s), the population rate is not higher than control in the first 400 ms after offset (difference -0.14 spikes/s, p = 0.565) or the next 400 ms (+0.10, p = 0.617), so there is no rebound on average. But more neurons are significantly excited (24) than suppressed (9) at p < 0.01 in the first 400 ms, the opposite of the balance the criterion requires; without the rate exclusion the counts are 30 and 9, and at p < 0.05 36 and 21. Counting every neuron by the sign of its change, more are lower (177) than higher (125), with a median change of -0.28 spikes/s, which fits the paper's "low level of sustained inhibition in a larger proportion"; whether the criterion's count at p < 0.01 is the right reading is asked of a reader.

## ed4-return-to-trajectory

Claim: neurons whose rates decrease during the delay epoch also recover after unilateral
photoinhibition, returning to the time-matched control trajectory and not decaying to the
end-point rate (27 neurons from 6 mice; paired t-test p < 0.001). Criterion: Experiment 1
pyramidal neurons with a lower rate at the end of the delay than at its start (two-tailed
t-test, 400 ms windows, trial types pooled, p < 0.05); rates normalized to each neuron's mean
rate, preferred and non-preferred types separately; per neuron, RMS over the time from
photoinhibition offset to the cue of (perturbed minus time-matched control) and of (perturbed
minus the control rate in the last 400 ms of the delay). Conditions (a) early delay (type 2),
(b) the three conditions pooled per neuron. The windows "start" and "end" of the delay are
read as its first and last 400 ms. Increasing neurons are shown as a variant. The RMS windows
start after photoinhibition has ended (verifier, round 1): 200 ms windows stamped at their centre
from 0.1 s after offset to 0.1 s before the cue, or 400 ms windows stamped at their end from 0.4 s
after offset to the cue; the end point is the control rate in the last 400 ms of the delay.

In [ ]:
def ed4_rows(direction, exp="unilateral", conds=(1, 2, 3), width=0.2):
    """Per neuron and condition, the RMS of (perturbed minus time-matched control) and of
    (perturbed minus the control rate in the last 400 ms of the delay), in units of the neuron's
    mean rate, preferred and non-preferred types. The rate is the `width` boxcar; 200 ms windows
    are stamped at their centre and taken from 0.1 s after offset to 0.1 s before the cue, 400 ms
    windows at their end and taken from 0.4 s after offset to the cue, so that no window overlaps
    photoinhibition or the response epoch."""
    rows = []
    for k, u in enumerate(units):
        if EXP[u["session"]] != exp or u["cell_type"] != "pyramidal" or M[k]["pref_counts"] is None:
            continue
        m = M[k] if width == 0.2 else MV(k, "code", width)
        s = SD[u["session"]]
        tr = u["trials"]
        st = s["stim_type"][tr]
        sd_ = alm3.side(s)[tr]
        ok = u["stable"] & ~s["trial_types"]["LickEarly"][tr] & (st == 0)
        if ok.sum() < 10:
            continue
        d0 = -1.3 if exp == "unilateral" else -1.7
        early = alm3.binned_window(u["binned"][ok], [d0 + 0.2], 0.4)[:, 0]
        late = alm3.binned_window(u["binned"][ok], [-0.2], 0.4)[:, 0]
        p = stats.ttest_ind(early, late).pvalue
        if not (p < 0.05) or (late.mean() < early.mean()) != (direction == "decrease"):
            continue
        norm = M[k]["fr_mean"] if M[k]["fr_mean"] > 0 else np.nan
        sgn = 1 if M[k]["pref_counts"][0] >= 0 else -1
        # end point: control rate in the last 400 ms of the delay, per trial type
        last = alm3.binned_window(u["binned"], [-0.2], 0.4)[:, 0] / 0.4
        endpoint = {typ: last[np.flatnonzero(ok & (sd_ == typ))].mean() / norm for typ in (1, -1)}
        stamp = GT if width == 0.2 else GT + width / 2
        r = {"k": k, "mouse": u["mouse"]}
        for c in conds:
            cd = m["cond"].get(c)
            if cd is None or cd["psth"][1] is None or cd["psth"][-1] is None or cd["nR"] < 3 or cd["nL"] < 3:
                r[c] = None
                continue
            win = (stamp >= OFFSET[c] + 0.1 - 1e-9) & (stamp <= -0.1 + 1e-9) if width == 0.2 else (stamp >= OFFSET[c] + 0.4 - 1e-9) & (stamp <= 1e-9)
            if not win.any():
                r[c] = None
                continue
            vals_tm, vals_ep = [], []
            for typ in (sgn, -sgn):
                ctrl = m["psth_ctrl_all"][typ] / norm
                pert = cd["psth"][typ] / norm
                vals_tm.append(pert[win] - ctrl[win])
                vals_ep.append(pert[win] - endpoint[typ])
            r[c] = (np.sqrt(np.mean(np.concatenate(vals_tm) ** 2)), np.sqrt(np.mean(np.concatenate(vals_ep) ** 2)))
        rows.append(r)
    return rows

for width in (0.2, 0.4):
    print(f"Windows of {int(width * 1000)} ms " + ("stamped at their centre, from 0.1 s after offset to 0.1 s before the cue:" if width == 0.2
                                                   else "stamped at their end, from 0.4 s after offset to the cue:"))
    for direction, exp, readings in (("decrease", "unilateral", (("(a) early delay", [2]), ("(b) pooled", [1, 2, 3]))),
                                     ("increase", "unilateral", (("(a) early delay", [2]), ("(b) pooled", [1, 2, 3]))),
                                     ("decrease", "exp2", (("variant, Experiment 2 ipsilateral", [4]),))):
        rows = ed4_rows(direction, exp, sorted({c for _, cs in readings for c in cs}), width=width)
        print(f"  {direction}, {exp}: {len(rows)} neurons from {len({r['mouse'] for r in rows})} mice")
        for lab, conds in readings:
            tm, ep = [], []
            for r in rows:
                v = [r[c] for c in conds if r.get(c) is not None]
                if v:
                    tm.append(np.mean([x[0] for x in v]))
                    ep.append(np.mean([x[1] for x in v]))
            tm, ep = np.array(tm), np.array(ep)
            t = stats.ttest_rel(tm, ep)
            print(f"    {lab}: n = {len(tm)}; RMS to time-matched control {tm.mean():.3f}, to end point {ep.mean():.3f}; "
                  f"paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}; smaller to time-matched in {(tm < ep).sum()} of {len(tm)}")
print("RMS in units of the neuron's mean rate; the end point is the control rate in the last 400 ms of the delay (criterion).")

**Result: not reproduced.** 55 Experiment 1 pyramidal neurons (4 mice) decrease their rate through the delay, against 27 from 6 mice. For the 47 with enough photoinhibition trials, the RMS distance to the time-matched control is not smaller than to the end point: with 200 ms windows 0.462 against 0.429 after early-delay photoinhibition (paired p = 0.27) and 0.500 against 0.494 pooled (p = 0.74); with 400 ms windows 0.378 against 0.367 (p = 0.65) and 0.451 against 0.470 (p = 0.52). In Experiment 2 (106 neurons, 7 mice) the two are equal (p = 0.92 and 0.43). For neurons with increasing rates the time-matched distance is smaller when the conditions are pooled (p = 0.041 and 0.049) and not significantly after early-delay photoinhibition (p = 0.086 and 0.28).

## ed10e-callosotomy-preparatory-activity

Claim: preparatory activity after callosotomy is similar to that of control mice, with 254 of 496
putative pyramidal neurons selective. Selectivity test of M3 on callosotomy control trials,
under the same readings as ed1-selective-fraction; "similar" read as the selective fraction
within 10 percentage points of the control mice's fraction under the same reading, and the
population delay selectivity present and rising toward the cue (criterion).

In [ ]:
pyr_c = (info.cell_type == "pyramidal") & (info.experiment == "ccx")
print("Callosotomy putative pyramidal neurons:", int(pyr_c.sum()), "(paper 496) from", info[pyr_c].mouse.nunique(), "mice,", info[pyr_c].session.nunique(), "sessions")
cc = info[info.experiment == "ccx"]
print("Callosotomy units by label:", cc.cell_type.value_counts().to_dict(), "; by width of the mean waveform:", cc.class_width.value_counts().to_dict(),
      "; upsampled:", cc.class_width_up.value_counts().to_dict())
rows = []
for trials_name, label in (("correct", "correct"), ("all", "correct+error")):
    for mask, grp in ((pyr_c, "callosotomy"), (pyr, "control mice")):
        P = np.array([M[i][f"p_sel_{trials_name}"] for i in np.flatnonzero(mask)])
        sig = P[:, [0, 1, 2]] < 0.05
        rows.append({"trials": label, "group": grp, "n": len(P), "selective": int(sig.any(1).sum()), "fraction": sig.any(1).mean(),
                     "sample": int(sig[:, 0].sum()), "delay": int(sig[:, 1].sum()), "response": int(sig[:, 2].sum())})
e10 = pd.DataFrame(rows)
print(e10.round(3).to_string(index=False))
print("Paper: 254 of 496 callosotomy pyramidal neurons selective; control 634 of 890.")
# population delay selectivity (preferred minus non-preferred, control test trials), selective neurons
for grp, exps in (("callosotomy", ["ccx"]), ("control mice, 1.7 s delay", ["exp2", "exp2b", "exp3"])):
    ks = [k for k, u in enumerate(units) if EXP[u["session"]] in exps and u["cell_type"] == "pyramidal" and M[k]["pref_counts"] is not None
          and selective(k, "correct")]
    sel = np.array([(1 if M[k]["pref_counts"][0] >= 0 else -1) * (M[k]["psth_ctrl"][1] - M[k]["psth_ctrl"][-1]) for k in ks])
    pts = [-1.5, -1.0, -0.5, -0.1]
    print(f"{grp}: {len(ks)} selective neurons; mean selectivity (spikes/s) at " + ", ".join(f"{t:+.1f} s: {sel[:, np.argmin(np.abs(GT - t))].mean():.2f}" for t in pts))

**Result: not reproduced.** The callosotomy sessions hold 519 units labelled pyramidal, against 496 (521 by the width of the mean waveform, also 521 after upsampling). With correct and error trials, 259 are selective, within 254 +- 25; with correct trials only, 303. The selective fraction is 0.50 against 0.64 in control mice under the same reading (0.58 against 0.71 with correct trials), more than the criterion's 10 points lower. Delay selectivity is present and rises toward the cue (1.08 spikes/s 1.5 s before the cue, 2.44 at 0.1 s before). The neuron count and the similarity of fractions fail, and the criterion has no partial rule.

## fig5-callosotomy-recovery

Claim: after callosotomy, ALM neurons recover their average spike rate after ipsilateral
photoinhibition but not their selectivity; selectivity recovers after contralateral
photoinhibition (217 +- 228 ms) and not after bilateral; selectivity along cd is reduced; 129
selective neurons, 16 sessions. Method of the code (analysis_compile_all_data_bilateralCCX.m) as
for fig2-bilateral-recovery; conditions as the data set labels them (types 10, 11, 12). The cd
part is tested in the population section below.

Every reading of M4 and of the selection is run, as the criterion asks (verifier, round 1):
the selection rule, "code" (selective in the sample or delay epoch with |preference count| > 1,
the scripts) or "M3" (selective in any of the sample, delay or response epochs, the claims file);
the screening trials, the code's, 10 of each type (M4 reading a) or 10 in total (M4 reading b);
the preference, from the whole count (M4 reading a) or from the epoch where the screening trials
differ most (M4 reading b); the trials of the selectivity test; the minimum trials per condition;
and the window. A condition counts as within tolerance only when at least half the resamples
reach 80% before the end of the search and their mean is within tolerance; the mean with the
other resamples counted at the end of the search is also shown.

In [ ]:
R5 = mean_rate_traces("ccx", [10, 11, 12])
rs5 = rate_summary(R5, [-1.3, -0.5, -0.3, -0.05])
print("Rates: photoinhibition over control population mean rate, and fraction of neurons with a significant change:")
print(rs5.drop(columns="label").round(3).to_string(index=False))
f5 = recovery_readings("ccx", [10, 11, 12], 0.0, {11: (217, 228)}, seed=51)
w5 = readings_wide(f5)
ok = f5.groupby(READING).agg(contra_within=("within_tol", lambda x: bool(x.dropna().all())))
w5 = w5.merge(ok.reset_index(), on=READING)
w5["ipsi_no_recovery"] = w5.frac_10 < 0.5
w5["bilateral_no_recovery"] = w5.frac_12 < 0.5
print("\nSelectivity recovery, every reading (10 ipsilateral, 11 contralateral, 12 bilateral):")
print(w5.round(2).to_string(index=False))
allok = w5.contra_within & w5.ipsi_no_recovery & w5.bilateral_no_recovery
print(f"\nReadings: {len(w5)}; contralateral within tolerance: {int(w5.contra_within.sum())}; ipsilateral not recovering: {int(w5.ipsi_no_recovery.sum())}; "
      f"bilateral not recovering: {int(w5.bilateral_no_recovery.sum())}; all three: {int(allok.sum())}")
print("Ipsilateral: fraction of resamples recovering", round(w5.frac_10.min(), 2), "to", round(w5.frac_10.max(), 2), "; deficit 0.2 s before the cue",
      round(w5.def_10.min(), 2), "to", round(w5.def_10.max(), 2))
print("Bilateral: fraction of resamples recovering at most", round(w5.frac_12.max(), 4), "; readings with at least one resample recovering:",
      int((w5.frac_12 > 0).sum()))
print("Paper: 129 neurons; contralateral 217 +- 228 ms; ipsilateral and bilateral selectivity do not recover.")

In [ ]:
def performance_table(exps, conds, by="mouse_pooled"):
    """Fraction correct (hits over hits and errors; lick-early and no-lick trials excluded, as in
    the data set's Demo_get_performance.m and the Methods) per mouse and condition, on performing
    trials. by = "mouse_pooled": trials pooled over a mouse's sessions (M7 a); "session_mean": the
    mean of its session values (M7 b)."""
    g = trials[trials.experiment.isin(exps) & trials.good & ~trials.lick_early & (trials.hit | trials.err) & trials.stim.isin([0] + conds)]
    if by == "mouse_pooled":
        t = g.groupby(["mouse", "stim"]).hit.mean().unstack()
    else:
        t = g.groupby(["mouse", "session", "stim"]).hit.mean().unstack().groupby("mouse").mean()
    n = g.groupby(["mouse", "stim"]).size().unstack()
    return t * 100, n

def perf_tests(t, conds, paired=True):
    rows = []
    for c in conds:
        d = t[[0, c]].dropna()
        test = stats.ttest_rel(d[c], d[0]) if paired else stats.ttest_ind(t[c].dropna(), t[0].dropna())
        rows.append({"condition": c, "label": alm3.STIM_LABELS[c], "mice": len(d), "control_%": d[0].mean(), "photoinhibition_%": d[c].mean(),
                     "drop": d[0].mean() - d[c].mean(), "t": test.statistic, "p": test.pvalue})
    return pd.DataFrame(rows)

## fig1d-earlier-photoinhibition-minimal

Claim: unilateral photoinhibition ending at least 0.3 s before the go cue (early and middle
delay) has minimal behavioural effect (p > 0.1). A claim of absence, judged on the paper's test:
per condition, performance with photoinhibition against control, two-tailed t-test across mice,
paired (M7 a; unpaired as a variant), at p < 0.05 (criterion). The paper's Fig. 1d comes from
behaviour-only sessions, which the data set does not hold; this uses the recording sessions of
Experiment 1 (types 1, 2, 3; the sessions with the no-ramp types -1 to -3 are left out, as the
data set says not to analyse them). The sample condition is reported as well.

In [ ]:
for by in ("mouse_pooled", "session_mean"):
    t, n = performance_table(["unilateral"], [1, 2, 3], by=by)
    print(f"Performance (% correct) per mouse, {by}:")
    print(t.round(1).to_string())
    for paired in (True, False):
        print(f"  {'paired' if paired else 'unpaired'} t-test against control:")
        print(perf_tests(t, [2, 3, 1], paired=paired).round(4).to_string(index=False))
print("\nTrials per mouse and condition:")
print(n.to_string())

**Result: reproduced.** In the Experiment 1 recording sessions (4 mice), early delay photoinhibition lowers performance by 5.0 points (paired t-test p = 0.23) and middle delay photoinhibition by 4.6 points (p = 0.22), neither significant at p < 0.05; both p-values are above the paper's 0.1. The other readings give p from 0.18 to 0.41. Sample photoinhibition changes performance by +0.6 points (p = 0.81). With four mice the test has little power.

## fig2b-bilateral-behaviour

Claim: unilateral photoinhibition in the early delay reduces performance only slightly (1 spot
70.3%, 4 spots 67.7%), bilateral photoinhibition severely (1 spot per side 58.0%) or to near
chance (4 spots per side 51.4%). Performance per condition over mice, tests of M7. The paper's
values come from 13 mice including behaviour-only sessions; here the recording sessions of the
bilateral experiments (types 4, 8, 7, 6), 0.8 s photoinhibition from delay onset.

In [ ]:
exps_b = ["exp2", "exp2b", "exp3"]
cmap = {4: 70.3, 8: 67.7, 7: 58.0, 6: 51.4}
for by in ("mouse_pooled", "session_mean"):
    t, n = performance_table(exps_b, [4, 8, 7, 6], by=by)
    print(f"Performance (% correct) per mouse, {by}:")
    print(t.round(1).to_string())
    for paired in (True, False):
        r = perf_tests(t, [4, 8, 7, 6], paired=paired)
        r["paper_%"] = r.condition.map(cmap)
        r["within_10%"] = (r["photoinhibition_%"] - r["paper_%"]).abs() <= 0.1 * r["paper_%"]
        print(f"  {'paired' if paired else 'unpaired'} t-tests against control:")
        print(r.round(4).to_string(index=False))
        d = t[[7, 8]].dropna()
        tt = stats.ttest_rel(d[7], d[8]) if paired else stats.ttest_ind(t[7].dropna(), t[8].dropna())
        print(f"  bilateral 1 spot against unilateral 4 spots: {d[7].mean():.1f} vs {d[8].mean():.1f} % over {len(d)} mice with both, t = {tt.statistic:.2f}, p = {tt.pvalue:.4f}")
print("\nTrials per mouse and condition:")
print(n.to_string())

**Result: partially reproduced.** In the bilateral recording sessions, performance (pooled per mouse) is 69.3% with unilateral 1-spot photoinhibition (10 mice, within 70.3 +- 7.0), 75.6% with unilateral 4 spots (3 mice, paper 67.7%), 65.8% with bilateral 1 spot (3 mice, paper 58.0%) and 58.1% with bilateral 4 spots (7 mice, paper 51.4%), against control 78.0%. Every condition is below control; the paired tests give p = 0.025, 0.15, 0.045 and 0.0001, and bilateral 1 spot against unilateral 4 spots p = 0.12. The order unilateral above bilateral 1 spot above bilateral 4 spots holds, but three values are outside tolerance and two tests are not significant.

## fig5b-callosotomy-behaviour

Claim: callosotomy leaves performance unaffected but makes it highly sensitive to unilateral
photoinhibition, with a significant interaction, dropping to the level of bilateral
photoinhibition in control mice (58.0%). The data set holds only sessions after the cut, so the
before-and-after comparison and the interaction cannot be tested; the after-cut test is
unilateral photoinhibition (type 10) against control (p < 0.01) and its performance against
58.0 +- 5.8% (criterion).

In [ ]:
print("Callosotomy sessions per mouse with dates:", ses[ses.experiment == "ccx"].groupby("mouse").session.apply(lambda x: f"{len(x)}: {x.str[-8:].min()} to {x.str[-8:].max()}").to_dict())
for by in ("mouse_pooled", "session_mean"):
    t, n = performance_table(["ccx"], [10, 11, 12], by=by)
    print(f"\nPerformance (% correct) per mouse, {by}:")
    print(t.round(1).to_string())
    for paired in (True, False):
        print(f"  {'paired' if paired else 'unpaired'} t-tests against control:")
        print(perf_tests(t, [10, 11, 12], paired=paired).round(4).to_string(index=False))
tc, _ = performance_table(exps_b, [4], by="mouse_pooled")
t5, _ = performance_table(["ccx"], [10], by="mouse_pooled")
print(f"\nControl-trial performance, control mice (bilateral experiments) {tc[0].mean():.1f}% over {tc[0].notna().sum()} mice; callosotomy mice {t5[0].mean():.1f}% over {t5[0].notna().sum()} mice; "
      f"unpaired t-test p = {stats.ttest_ind(tc[0].dropna(), t5[0].dropna()).pvalue:.3f} (different mice; not the paper's before/after test)")
print(f"Ipsilateral 1 spot: control mice {tc[4].mean():.1f}%, callosotomy mice {t5[10].mean():.1f}%")

**Result: partially reproduced.** The data set holds only sessions after the cut, so the before-and-after comparison and the interaction cannot be tested. After the cut, unilateral (ipsilateral 1 spot) photoinhibition lowers performance from 74.6% to 59.1% (7 mice, paired p = 0.0033), and 59.1% is within 58.0 +- 5.8%.

In [ ]:
# Population analyses (Methods, Analysis of population dynamics in the activity space; code,
# analysis_compile_all_data_bilateral_activityModes.m).
def tidx(t):
    return int(np.argmin(np.abs(alm3.POP_T - t)))

I_LATE = tidx(-0.4)   # window [-0.4, 0]: last 400 ms of the delay

def cd_windows(reading):
    """Windows (by start time) counted as in the sample and delay epochs (-3.0 to 0 s from the
    cue, 1.3 s sample and 1.7 s delay): (b) wholly inside, as the code (start > -3, <= -0.4);
    (a) centre inside."""
    if reading == "inside":
        return np.flatnonzero((alm3.POP_T > -3.0 - 1e-9) & (alm3.POP_T <= -0.4 + 1e-9))
    return np.flatnonzero((alm3.POP_T + 0.2 >= -3.0 - 1e-9) & (alm3.POP_T + 0.2 <= 0 + 1e-9))

def cd_vector(P, train_R, train_L, kind, win_reading):
    X = P["X"]
    win = cd_windows(win_reading)
    if kind == "meandiff":
        w = (X[train_R][:, :, win].mean(0) - X[train_L][:, :, win].mean(0)).mean(1)
    else:  # the code's LDA per window, averaged
        w = np.mean([alm3.lda_direction(X[train_R][:, :, i], X[train_L][:, :, i]) for i in win[::5]], axis=0)
    return w / np.linalg.norm(w)

def cd_projections(P, kind="meandiff", win_reading="inside", w=None):
    """Projection of every trial on cd at every window, minus the decision boundary in the last
    400 ms of the delay, from correct training trials."""
    hit = P["hit"]
    tR = P["train"][1][hit[P["train"][1]]]
    tL = P["train"][-1][hit[P["train"][-1]]]
    if w is None:
        w = cd_vector(P, tR, tL, kind, win_reading)
    proj = np.einsum("tnw,n->tw", P["X"], w)
    b = alm3.boundary(proj[tR, I_LATE], proj[tL, I_LATE])
    return proj - b, w, (tR, tL)

def cd_projections_loo(P, win_reading="inside"):
    """Leave-one-out projections (M6 reading b): each control trial is projected on a cd and a
    boundary computed without it; photoinhibition trials on the cd of all correct control trials."""
    X = P["X"]
    hit = P["hit"]
    win = cd_windows(win_reading)
    tR = P["train"][1][hit[P["train"][1]]]
    tL = P["train"][-1][hit[P["train"][-1]]]
    proj, w, _ = cd_projections(P, "meandiff", win_reading)
    mR = X[tR][:, :, win].mean(axis=(0, 2)) * len(tR)
    mL = X[tL][:, :, win].mean(axis=(0, 2)) * len(tL)
    xi_win = X[:, :, win].mean(2)
    for i in np.r_[P["test"][1], P["test"][-1]]:
        r_, l_ = tR, tL
        sR, sL = mR, mL
        if i in set(tR):
            r_ = tR[tR != i]
            sR = mR - xi_win[i]
        elif i in set(tL):
            l_ = tL[tL != i]
            sL = mL - xi_win[i]
        wi = sR / len(r_) - sL / len(l_)
        wi /= np.linalg.norm(wi)
        b = alm3.boundary(X[r_][:, :, I_LATE] @ wi, X[l_][:, :, I_LATE] @ wi)
        proj[i] = X[i].T @ wi - b
    return proj

def session_rocs(P, proj, conds, t_index=I_LATE, average=None):
    out = {}
    def val(idx):
        return proj[idx][:, average].mean(1) if average is not None else proj[idx, t_index]
    out[0] = alm3.auc(val(P["test"][1]), val(P["test"][-1]))
    for c in conds:
        out[c] = alm3.auc(val(P["groups"][(c, 1)]), val(P["groups"][(c, -1)]))
    return out

def build_population(exp, conds, cell_types=("pyramidal",), split="code", seed=7, min_group=5):
    pops = []
    for i, name in enumerate(ses[ses.experiment == exp].session):
        su = [u for u in units if u["session"] == name]
        if not su:
            continue
        P = alm3.population_session(SD[name], su, conds, cell_types=cell_types, seed=seed + i, split=split, min_group=min_group)
        if P is not None:
            pops.append(P)
    return pops

POP = {}
for exp, conds in (("exp2", [4, 5, 6]), ("exp3", [7, 8, 9]), ("exp2b", [4, 5, 7]), ("ccx", [10, 11, 12])):
    for cells in ("pyramidal", "all"):
        for split in ("code", "half", "loo"):
            ct = ("pyramidal",) if cells == "pyramidal" else ("pyramidal", "FS", "none")
            POP[(exp, cells, split)] = build_population(exp, conds, ct, split)
for k, v in POP.items():
    print(k, "sessions", len(v), "mice", len({p["mouse"] for p in v}), "neurons", sum(p["n_neurons"] for p in v))
# Reading of M6 with more than 5 trials in every group (the Methods' "> 5 trials per condition");
# the code keeps groups of 5.
POP6 = {exp: build_population(exp, conds, min_group=6) for exp, conds in (("exp2", [4, 5, 6]), ("ccx", [10, 11, 12]))}
for exp, v in POP6.items():
    print(f"{exp}, pyramidal, code split, more than 5 trials per group: sessions {len(v)}, mice {len({p['mouse'] for p in v})}")

## fig3-cd-roc

Claim: along the coding direction, trajectories recover after ipsilateral photoinhibition (ROC
at the end of the delay 0.73 +- 0.02 against control 0.76 +- 0.03), are little affected by
contralateral photoinhibition (0.74 +- 0.03) and stay collapsed after bilateral photoinhibition
(0.58 +- 0.03); 16 sessions. Readings run: neurons pyramidal only (code) or all units; split of
control trials as the code or random halves (M6 a); cd as the mean difference of the Methods
with windows wholly inside (code) or centred in the sample and delay, and the code's LDA as a
variant. ROC: area under the ROC curve of lick right against lick left projections in the last
400 ms of the delay, control test trials and each photoinhibition condition, per session,
mean +- s.e.m. over sessions.

In [ ]:
def roc_table(exp, conds, kinds=(("meandiff", "inside"), ("meandiff", "centre"), ("lda", "inside")), **kw):
    rows = []
    for (e, cells, split), pops in POP.items():
        if e != exp:
            continue
        for kind, wr in kinds:
            if split == "loo" and kind != "meandiff":
                continue
            R = []
            for P in pops:
                proj = cd_projections_loo(P, wr) if split == "loo" else cd_projections(P, kind, wr)[0]
                R.append(session_rocs(P, proj, conds, **kw))
            R = pd.DataFrame(R)
            r = {"cells": cells, "split": split, "cd": f"{kind}/{wr}", "sessions": len(R)}
            for c in [0] + conds:
                r[f"roc_{c}"] = R[c].mean()
                r[f"sem_{c}"] = R[c].std(ddof=1) / np.sqrt(len(R))
            rows.append(r)
    return pd.DataFrame(rows)

rt3 = roc_table("exp2", [4, 5, 6])
exp3v = {0: (0.76, 0.03), 4: (0.73, 0.02), 5: (0.74, 0.03), 6: (0.58, 0.03)}
rt3["all_within"] = np.all([(rt3[f"roc_{c}"] - v).abs() <= tol for c, (v, tol) in exp3v.items()], axis=0)
rt3["order"] = (rt3[["roc_0", "roc_4", "roc_5"]].min(1) - rt3["roc_6"]) > 0.1
print(rt3.round(3).to_string(index=False))
print("Paper: 16 sessions; control 0.76 +- 0.03, ipsilateral 0.73 +- 0.02, contralateral 0.74 +- 0.03, bilateral 0.58 +- 0.03.")
P0 = POP[("exp2", "pyramidal", "code")]
print("\nSessions (code selection, pyramidal):", [(p["session"], p["n_neurons"]) for p in P0])
R = pd.DataFrame([session_rocs(P, cd_projections(P)[0], [4, 5, 6]) for P in P0])
R["mouse"] = [p["mouse"] for p in P0]
print("Per mouse (mean over sessions), mean-difference cd, code split:")
print(R.groupby("mouse").mean().round(3).to_string())
R6 = pd.DataFrame([session_rocs(P, cd_projections(P)[0], [4, 5, 6]) for P in POP6["exp2"]])
print(f"\nReading with more than 5 trials per group ({len(R6)} sessions), mean-difference cd, code split, pyramidal: "
      + ", ".join(f"{c}: {R6[c].mean():.3f}" for c in [0, 4, 5, 6]))

**Result: reproduced.** The session rule gives the paper's 16 sessions from 7 mice. With the leave-one-out split of control trials (M6 reading b), the mean-difference cd and pyramidal neurons, the end-of-delay ROC is 0.731 +- 0.025 (control), 0.713 +- 0.023 (ipsilateral), 0.742 +- 0.033 (contralateral) and 0.592 +- 0.033 (bilateral), all within the paper's 0.76 +- 0.03, 0.73 +- 0.02, 0.74 +- 0.03 and 0.58 +- 0.03; windows centred in the epochs give the same. With the code's split the control and ipsilateral values fall just below tolerance (0.719 and 0.706), and with random halves the contralateral value is 0.684; with all units the ipsilateral value is 0.667 to 0.681 in every split.
With more than 5 trials in every group (the Methods' wording; the code keeps groups of 5), 12 sessions remain and the control ROC is 0.681, outside tolerance; the result rests on the code's session rule, which gives the paper's 16 sessions.

## ed8-roc-during-photoinhibition

Claim: during photoinhibition the cd separation falls (ROC 0.54 +- 0.03 ipsilateral, 0.64 +- 0.03
contralateral, 0.54 +- 0.01 bilateral, against control 0.72 +- 0.02). Window (a) the 400 ms
window ending at the start of the offset ramp (1.5 to 1.1 s before the cue), (b) the projection
averaged over the windows inside the 0.8 s photoinhibition (criterion). Other readings as fig3-cd-roc.

In [ ]:
exp8 = {0: (0.72, 0.02), 4: (0.54, 0.03), 5: (0.64, 0.03), 6: (0.54, 0.01)}
during_avg = np.flatnonzero((alm3.POP_T >= -1.7 - 1e-9) & (alm3.POP_T <= -1.3 + 1e-9))
tabs = []
for lab, kw in (("(a) 400 ms ending at ramp start", {"t_index": tidx(-1.5)}), ("(b) average over photoinhibition", {"average": during_avg})):
    t = roc_table("exp2", [4, 5, 6], **kw)
    t.insert(0, "window", lab)
    tabs.append(t)
rt8 = pd.concat(tabs, ignore_index=True)
rt8["all_within"] = np.all([(rt8[f"roc_{c}"] - v).abs() <= tol for c, (v, tol) in exp8.items()], axis=0)
print(rt8.round(3).to_string(index=False))
print("Paper: control 0.72 +- 0.02, ipsilateral 0.54 +- 0.03, contralateral 0.64 +- 0.03, bilateral 0.54 +- 0.01.")

**Result: reproduced.** With projections averaged over the photoinhibition period (reading b), all units and the code's split, the ROC is 0.703 (control), 0.534 (ipsilateral), 0.642 (contralateral) and 0.539 (bilateral), all within the paper's 0.72 +- 0.02, 0.54 +- 0.03, 0.64 +- 0.03 and 0.54 +- 0.01; three of the 32 readings match. With pyramidal neurons only the bilateral value is 0.51 and outside its narrow tolerance. With window (a) the ipsilateral values are 0.490 to 0.544 and the bilateral values 0.473 to 0.526, and no reading matches.

## ed7e-size-and-roc

Claim: recovery of selectivity after unilateral photoinhibition is less robust with larger
photoinhibition, and cd trajectories separate at the end of the delay with ROC 0.80 +- 0.08
(control), 0.64 +- 0.10 (ipsilateral 4 spots), 0.68 +- 0.15 (contralateral 4 spots) and 0.54
(bilateral 1 spot; tolerance 0.054, 0.08 as a variant). Experiment 3 sessions with > 5 neurons
and enough trials (the session rule of fig3-cd-roc); "less robust" judged by the ipsilateral 4-spot
ROC below the Experiment 2 ipsilateral 1-spot ROC.

In [ ]:
rt7 = roc_table("exp3", [7, 8, 9])
exp7 = {0: (0.80, 0.08), 8: (0.64, 0.10), 9: (0.68, 0.15), 7: (0.54, 0.054)}
rt7["all_within"] = np.all([(rt7[f"roc_{c}"] - v).abs() <= tol for c, (v, tol) in exp7.items()], axis=0)
rt7["all_within_bilat_tol_0.08"] = np.all([(rt7[f"roc_{c}"] - v).abs() <= (0.08 if c == 7 else tol) for c, (v, tol) in exp7.items()], axis=0)
print(rt7.round(3).to_string(index=False))
m = rt3.set_index(["cells", "split", "cd"])["roc_4"]
rt7["exp2_ipsi_1spot"] = [m.get((r.cells, r.split, r.cd)) for r in rt7.itertuples()]
print("\nIpsilateral 4 spots (Experiment 3) against ipsilateral 1 spot (Experiment 2):")
print(rt7[["cells", "split", "cd", "roc_8", "exp2_ipsi_1spot"]].round(3).to_string(index=False))
print("Sessions (code selection, pyramidal):", [(p["session"], p["n_neurons"], p["counts"]) for p in POP[("exp3", "pyramidal", "code")]])
print("Experiment 3 sessions in the data set:", ses[ses.experiment == "exp3"].session.tolist())
# Variant for the question to a reader (ED Fig. 2b gives n = 102 for Experiment 3 bilateral 1 spot): the bilateral 1-spot
# sessions with types 4, 5, 7 pooled with Experiment 3 for the bilateral ROC.
rows = []
for cells in ("pyramidal", "all"):
    for split in ("code", "half", "loo"):
        vals = []
        for exp, conds in (("exp3", [7, 8, 9]), ("exp2b", [4, 5, 7])):
            for P in POP[(exp, cells, split)]:
                proj = cd_projections_loo(P) if split == "loo" else cd_projections(P)[0]
                vals.append(session_rocs(P, proj, conds)[7])
        rows.append({"cells": cells, "split": split, "sessions": len(vals), "roc_7_pooled": np.mean(vals), "within_0.054": abs(np.mean(vals) - 0.54) <= 0.054})
print("\nBilateral 1 spot, Experiment 3 and the sessions with types 4, 5, 7 pooled (mean-difference cd):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced.** Four of the six Experiment 3 sessions pass the session rule. With the mean-difference cd and pyramidal neurons, the ROC is 0.762 (control), 0.621 (ipsilateral 4 spots) and 0.702 (contralateral 4 spots), within tolerance, and 0.604 for bilateral 1 spot, outside 0.54 +- 0.054; no reading puts the bilateral value within 0.054 (0.598 to 0.642), and it is within only the 0.08 variant. The ipsilateral 4-spot ROC is below the Experiment 2 ipsilateral 1-spot ROC (0.621 against 0.706), so recovery is less robust with the larger photoinhibition. With the bilateral 1-spot sessions holding types 4, 5 and 7 pooled (9 sessions), the bilateral ROC is 0.570 to 0.602, within 0.054 in 4 of 6 readings; whether the paper pooled them is asked of a reader.

## ed9b-cd-stability

Claim: the direction of the lick right minus lick left response difference is stable through the
sample and delay epochs (correlation between late sample and late delay 0.61 +- 0.05). Per
session of fig3-cd-roc, Pearson correlation of w in the last 400 ms of the sample epoch (1.7 to
2.1 s before the cue) and in the last 400 ms of the delay, correct control trials (criterion);
all correct control trials (a reading) or the correct training trials (as for cd).

In [ ]:
rows = []
for (e, cells, split), pops in POP.items():
    if e != "exp2" or split != "code":
        continue
    for trial_set in ("all correct control", "correct training"):
        rr = []
        for P in pops:
            hit = P["hit"]
            if trial_set == "all correct control":
                R_ = np.r_[P["train"][1], P["test"][1]]
                L_ = np.r_[P["train"][-1], P["test"][-1]]
            else:
                R_, L_ = P["train"][1], P["train"][-1]
            R_, L_ = R_[hit[R_]], L_[hit[L_]]
            X = P["X"]
            w1 = X[R_][:, :, tidx(-2.1)].mean(0) - X[L_][:, :, tidx(-2.1)].mean(0)
            w2 = X[R_][:, :, I_LATE].mean(0) - X[L_][:, :, I_LATE].mean(0)
            rr.append(np.corrcoef(w1, w2)[0, 1])
        rr = np.array(rr)
        rows.append({"cells": cells, "trials": trial_set, "sessions": len(rr), "mean_r": np.nanmean(rr), "sem": np.nanstd(rr, ddof=1) / np.sqrt(np.isfinite(rr).sum()),
                     "within_0.61+-0.05": abs(np.nanmean(rr) - 0.61) <= 0.05})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("Paper: 0.61 +- 0.05.")

**Result: reproduced.** Over all correct control trials, the correlation of the lick right minus lick left difference between the last 400 ms of the sample and of the delay is 0.590 +- 0.075 over 16 sessions (0.622 with all units), within 0.61 +- 0.05.

## ed8a-mode-variance

Claim: the projection on cd captures 65.6 +- 5.1% of population selectivity and 36.4 +- 6.3% of
task-related activity variance over the sample and delay epochs; modes 1 to 5 together explain
98.5 +- 0.5% of the variance and 95.8 +- 1.2% of the selectivity. Modes as in the Methods and
the code: mode 1 cd; modes 2 and 3 the ipsilateral perturbed minus unperturbed mean response in
the last 400 ms of the delay, lick right and lick left; modes 4 and 5 the first singular vectors
of the baseline-subtracted lick right and lick left mean responses concatenated over the sample
and delay windows; each Gram-Schmidt orthogonal to the earlier ones. Training control trials
(code). Baseline (a) the window before the sample (3.4 to 3.0 s before the cue, the code's
`time_bins <= -3.4`), (b) the mean over the trial. Share (a) RMS ratio, (b) squared norms (the
code). Total variance and selectivity: the squared norm over the full neuron space (code). cd
as the mean difference (Methods) or LDA (code). The shares use only the training trials, so the
leave-one-out split (M6 b) amounts to using all control trials.

In [ ]:
def gram_schmidt(vs):
    out = []
    for v in vs:
        v = v.astype(float).copy()
        for o in out:
            v -= (v @ o) * o
        n = np.linalg.norm(v)
        if n > 1e-12:
            out.append(v / n)
    return out

SD_WIN = np.flatnonzero((alm3.POP_T > -3.0 - 1e-9) & (alm3.POP_T <= -0.4 + 1e-9))

def modes_for(P, kind="meandiff", baseline="pre", trials="all"):
    """Modes 1 to 5 (Methods). trials "all": modes 2, 3 and the PSTHs of modes 4, 5 from correct
    and error trials grouped by instructed movement (code); "correct": from correct trials only
    (criterion)."""
    X = P["X"]
    trR, trL = P["train"][1], P["train"][-1]
    _, w1, _ = cd_projections(P, kind, "inside")
    iR, iL = P["groups"][(4, 1)], P["groups"][(4, -1)]
    if trials == "correct":
        hit = P["hit"]
        trR, trL, iR, iL = trR[hit[trR]], trL[hit[trL]], iR[hit[iR]], iL[hit[iL]]
    w2 = X[iR][:, :, I_LATE].mean(0) - X[trR][:, :, I_LATE].mean(0)
    w3 = X[iL][:, :, I_LATE].mean(0) - X[trL][:, :, I_LATE].mean(0)
    R = X[trR][:, :, SD_WIN].mean(0)
    L = X[trL][:, :, SD_WIN].mean(0)
    if baseline == "pre":
        base = X[np.r_[trR, trL]][:, :, tidx(-3.4)].mean(0)
    else:
        base = X[np.r_[trR, trL]].mean(axis=(0, 2))
    A = np.concatenate([R, L], axis=1) - base[:, None]
    u, sv, vt = np.linalg.svd(A, full_matrices=False)
    basis = gram_schmidt([w1, w2, w3] + [u[:, i] for i in range(u.shape[1])])
    return basis, A, R - L

rows = []
for cells, split in [(c, sp) for c in ("pyramidal", "all") for sp in ("code", "half", "loo")]:
    pops = POP[("exp2", cells, split)]
    for (kind, trials_m) in (("meandiff", "all"), ("meandiff", "correct"), ("lda", "all"), ("lda", "correct")):
        for baseline in ("pre", "trial"):
            vals = []
            for P in pops:
                basis, A, S = modes_for(P, kind, baseline, trials_m)
                tot_v, tot_s = (A ** 2).sum(), (S ** 2).sum()
                pv = [((A.T @ b) ** 2).sum() / tot_v for b in basis[:5]]
                ps = [((S.T @ b) ** 2).sum() / tot_s for b in basis[:5]]
                vals.append({"var_cd": pv[0], "sel_cd": ps[0], "var_1_5": sum(pv), "sel_1_5": sum(ps),
                             "var_cd_rms": np.sqrt(pv[0]), "sel_cd_rms": np.sqrt(ps[0]), "var_1_5_rms": np.sqrt(sum(pv)), "sel_1_5_rms": np.sqrt(sum(ps)),
                             "var_mode4": pv[3], "n": P["n_neurons"]})
            v = pd.DataFrame(vals)
            for share in ("squared", "rms"):
                suf = "" if share == "squared" else "_rms"
                r = {"cells": cells, "split": split, "cd": kind, "mode_trials": trials_m, "baseline": baseline, "share": share, "sessions": len(v)}
                for q in ("sel_cd", "var_cd", "var_1_5", "sel_1_5"):
                    r[q] = 100 * v[q + suf].mean()
                    r[q + "_sem"] = 100 * v[q + suf].std(ddof=1) / np.sqrt(len(v))
                rows.append(r)
e8a = pd.DataFrame(rows)
expa = {"sel_cd": (65.6, 5.1), "var_cd": (36.4, 6.3), "var_1_5": (98.5, 0.5), "sel_1_5": (95.8, 1.2)}
e8a["all_within"] = np.all([(e8a[q] - v).abs() <= t for q, (v, t) in expa.items()], axis=0)
e8a["cd_within"] = np.all([(e8a[q] - v).abs() <= t for q, (v, t) in list(expa.items())[:2]], axis=0)
print(e8a.round(1).to_string(index=False))
print("Paper: selectivity on cd 65.6 +- 5.1%, variance on cd 36.4 +- 6.3%, modes 1-5: variance 98.5 +- 0.5%, selectivity 95.8 +- 1.2%.")
print("Neurons per session (pyramidal):", [p["n_neurons"] for p in POP[("exp2", "pyramidal", "code")]])

**Result: partially reproduced.** With the mean-difference cd of the Methods and random halves of the control trials (M6 reading a), cd captures 67.1% of the selectivity and 39.9% of the variance (pyramidal neurons, pre-sample baseline, squared-norm shares, modes from correct and error trials), both within the paper's 65.6 +- 5.1% and 36.4 +- 6.3%. Modes 1 to 5 capture 96.3% of the variance and 92.5% of the selectivity, outside 98.5 +- 0.5% and 95.8 +- 1.2%; no reading puts all four values within tolerance, also with modes 2 and 3 and the mean responses from correct trials only (pyramidal, random halves: 96.8% and 94.5% for modes 1 to 5, with cd at 72.8% of the selectivity, outside). With the code's split the cd selectivity share is 72.5%, outside; with the code's discriminant as cd it is 66.1%.

## fig3-cd-predicts-movement

Claim: deviations along the coding direction predict the upcoming movement trial by trial, in
control trials and after unilateral and bilateral perturbation. Trials pooled over the sessions
of fig3-cd-roc (Methods); per condition, the fraction of trials whose lick (the instructed side
on correct trials, the other side on errors) is on the side of the decision boundary where the
projection lies, with a binomial test against 0.5, and a logistic regression of lick right on
the signed projection (criterion). Control trials are the test trials. cd of unit length
(reading b of the scaling; the tests do not depend on the scale).

In [ ]:
import statsmodels.api as sm

def pooled_projections(pops, conds, kind="meandiff", which="cd", t_index=I_LATE):
    rows = []
    for P in pops:
        if which == "cd":
            proj, _, _ = cd_projections(P, kind, "inside")
        else:
            proj = which(P)
        lick_right = np.where(P["hit"], P["side"] == 1, P["side"] == -1)
        for c in [0] + conds:
            idx = np.r_[P["test"][1], P["test"][-1]] if c == 0 else np.r_[P["groups"][(c, 1)], P["groups"][(c, -1)]]
            for i in idx:
                rows.append({"session": P["session"], "mouse": P["mouse"], "condition": c, "proj": proj[i, t_index],
                             "lick_right": bool(lick_right[i]), "instructed_right": P["side"][i] == 1, "hit": bool(P["hit"][i])})
    return pd.DataFrame(rows)

def prediction_table(df):
    rows = []
    for c, d in df.groupby("condition"):
        pred = d.proj > 0
        correct = int((pred == d.lick_right).sum())
        b = stats.binomtest(correct, len(d), 0.5, alternative="greater")
        lr = sm.Logit(d.lick_right.astype(float), sm.add_constant(d.proj.values)).fit(disp=0)
        rows.append({"condition": c, "label": "control" if c == 0 else alm3.STIM_LABELS[c], "trials": len(d), "accuracy": correct / len(d),
                     "binom_p": b.pvalue, "logit_slope": np.asarray(lr.params)[1], "logit_p": np.asarray(lr.pvalues)[1],
                     "accuracy_error_trials": ((d.proj > 0) == d.lick_right)[~d.hit].mean(), "error_trials": int((~d.hit).sum())})
    return pd.DataFrame(rows)

for cells in ("pyramidal", "all"):
    dfp = pooled_projections(POP[("exp2", cells, "code")], [4, 5, 6])
    print(f"Neurons: {cells}")
    print(prediction_table(dfp).to_string(index=False, float_format=lambda x: f"{x:.3g}"))
dfp = pooled_projections(POP[("exp2", "pyramidal", "code")], [4, 5, 6])
print("\nFraction of lick right responses by projection bin (control and bilateral, pyramidal):")
dfp["bin"] = pd.cut(dfp.proj, [-np.inf, -10, -5, -2, 0, 2, 5, 10, np.inf])
print(dfp[dfp.condition.isin([0, 6])].groupby(["condition", "bin"], observed=True).lick_right.agg(["mean", "size"]).round(2).to_string())
print("Projection range (spikes/s along unit-length cd):", round(dfp.proj.min(), 1), "to", round(dfp.proj.max(), 1))

**Result: reproduced.** On trials pooled over the 16 sessions, the side of the decision boundary predicts the lick on 72.8% of control, 68.6% of ipsilateral, 68.4% of contralateral and 67.1% of bilateral trials (binomial p below 1e-6 in each), and the logistic regression slope of lick right on the projection is positive with p below 1e-6 in each condition. After bilateral photoinhibition the fraction of lick right responses is 0.33 in the most negative projection bin and 0.77 in the most positive.

## fig3-other-modes

Claim: the persistent mode is altered by perturbation but neither discriminates trial type nor
predicts behaviour; the ramping mode shows non-selective ramping, is resistant to perturbation
and does not predict behaviour. Modes 2 (persistent, lick right) and 4 (ramping) of
ed8a-mode-variance, mean-difference cd, pre-sample baseline; decision boundaries as for cd and
signs so that lick right projects higher on training trials (code). Criterion: ROC in the last
400 ms of the delay within 0.5 +- 0.1 over sessions and control prediction accuracy with binomial
p > 0.05; ramping-mode projection for ipsilateral and bilateral trials within the s.e.m. band of
the control projection over sessions, persistent-mode projection outside it for ipsilateral.
Splits: the code's and random halves (M6 a); the leave-one-out split (M6 b) is not run here, as
it needs all five modes recomputed for every control trial.

In [ ]:
def mode_projection(P, mode_index, kind="meandiff"):
    basis, _, _ = modes_for(P, kind, "pre")
    w = basis[mode_index]
    proj = np.einsum("tnw,n->tw", P["X"], w)
    hit = P["hit"]
    tR = P["train"][1][hit[P["train"][1]]]
    tL = P["train"][-1][hit[P["train"][-1]]]
    if proj[tR, I_LATE].mean() < proj[tL, I_LATE].mean():
        proj = -proj
    return proj - alm3.boundary(proj[tR, I_LATE], proj[tL, I_LATE]), proj

for cells, split in (("pyramidal", "code"), ("all", "code"), ("pyramidal", "half"), ("all", "half")):
    pops = POP[("exp2", cells, split)]
    print(f"Neurons: {cells}, split: {split}")
    for mi, name in ((1, "persistent (mode 2)"), (3, "ramping (mode 4)"), (0, "cd (mode 1), for comparison")):
        rocs, raw = [], []
        for P in pops:
            pr, praw = mode_projection(P, mi)
            rocs.append(session_rocs(P, pr, [4, 5, 6]))
            # mean raw projection in the last 400 ms of the delay, per condition, both trial types
            r = {0: praw[np.r_[P["test"][1], P["test"][-1]], I_LATE].mean()}
            for c in (4, 5, 6):
                r[c] = praw[np.r_[P["groups"][(c, 1)], P["groups"][(c, -1)]], I_LATE].mean()
            raw.append(r)
        rocs, raw = pd.DataFrame(rocs), pd.DataFrame(raw)
        dfm = pooled_projections(pops, [4, 5, 6], which=lambda P, mi=mi: mode_projection(P, mi)[0])
        pt = prediction_table(dfm).set_index("condition")
        diff = raw[[4, 5, 6]].sub(raw[0], axis=0)
        sem0 = raw[0].std(ddof=1) / np.sqrt(len(raw))
        print(f"  {name}: ROC control {rocs[0].mean():.3f} +- {rocs[0].std(ddof=1) / np.sqrt(len(rocs)):.3f}, ipsi {rocs[4].mean():.3f}, contra {rocs[5].mean():.3f}, bilateral {rocs[6].mean():.3f}; "
              f"control accuracy {pt.loc[0, 'accuracy']:.3f} (binomial p {pt.loc[0, 'binom_p']:.3g}); "
              f"perturbed minus control projection (mean over sessions): ipsi {diff[4].mean():+.2f}, contra {diff[5].mean():+.2f}, bilateral {diff[6].mean():+.2f}; "
              f"control s.e.m. band {sem0:.2f}")

**Result: not reproduced.** The persistent and ramping modes discriminate trial type only weakly (control ROC 0.567 and 0.577), within 0.5 +- 0.1, but both predict the lick on control trials above chance (accuracy 0.568 and 0.570, binomial p = 0.003 and 0.002). The ramping mode is little changed by ipsilateral and bilateral photoinhibition (+0.51 and -0.48 against a control band of 1.00), but the persistent mode's ipsilateral change (-2.09) is also inside its band (2.17).

## ed6-bilateral-bias-inconsistent

Claim: bilateral photoinhibition biases movements inconsistently across mice and sessions, and
the direction of a session's bias follows which control trajectory its perturbed cd
trajectories approach. Sessions of fig3-cd-roc, bilateral photoinhibition (type 6). Per session,
bias = performance on lick right minus lick left trials under photoinhibition; biased when
Fisher's exact test of the two performances gives p < 0.05; side of the mean perturbed
projection in the last 400 ms of the delay (a) relative to the decision boundary, (b) nearer of
the mean control lick right and lick left projections. Reproduced when biased sessions of both
signs occur and the sign agrees with the side in at least 80% of biased sessions (criterion).

In [ ]:
for cells in ("pyramidal", "all"):
    rows = []
    for P in POP[("exp2", cells, "code")]:
        proj, _, _ = cd_projections(P)
        gR, gL = P["groups"][(6, 1)], P["groups"][(6, -1)]
        hR, hL = P["hit"][gR], P["hit"][gL]
        bias = hR.mean() - hL.mean()
        p = stats.fisher_exact([[hR.sum(), (~hR).sum()], [hL.sum(), (~hL).sum()]]).pvalue
        mp = proj[np.r_[gR, gL], I_LATE].mean()
        cR, cL = proj[P["test"][1], I_LATE].mean(), proj[P["test"][-1], I_LATE].mean()
        rows.append({"session": P["session"], "mouse": P["mouse"], "perf_R": hR.mean(), "perf_L": hL.mean(), "bias": bias, "fisher_p": p,
                     "mean_proj": mp, "side_a": np.sign(mp), "side_b": 1 if abs(mp - cR) < abs(mp - cL) else -1})
    b6 = pd.DataFrame(rows)
    biased = b6[b6.fisher_p < 0.05]
    print(f"Neurons: {cells}")
    print(b6.round(3).to_string(index=False))
    for s_ in ("side_a", "side_b"):
        agree = (np.sign(biased.bias) == biased[s_]).mean() if len(biased) else np.nan
        print(f"  biased sessions {len(biased)} (lick right bias {int((biased.bias > 0).sum())}, lick left bias {int((biased.bias < 0).sum())}); sign agrees with {s_}: {agree:.2f}")
    allc = (np.sign(b6.bias) == b6.side_a).mean()
    print(f"  all sessions: sign of bias agrees with side (a) in {allc:.2f}; Spearman of bias with mean projection r = {stats.spearmanr(b6.bias, b6.mean_proj).statistic:.2f}, p = {stats.spearmanr(b6.bias, b6.mean_proj).pvalue:.3g}")

**Result: reproduced.** Under bilateral photoinhibition 8 of 16 sessions are biased (Fisher p < 0.05), 6 toward lick right and 2 toward lick left. The sign of the bias agrees with the nearer control trajectory (reading b) in 88% of biased sessions, and with the side of the decision boundary (reading a) in 62%.

## fig5-callosotomy-recovery: selectivity along cd

The cd part of fig5-callosotomy-recovery: the ipsilateral cd ROC at the end of the delay in the
callosotomy sessions (session rule of fig3-cd-roc), against (a) the callosotomy control ROC and
(b) the Experiment 2 ipsilateral ROC; reduced read as lower by a paired (a) or unpaired (b)
t-test over sessions at p < 0.05, a standard choice.

In [ ]:
rt5 = roc_table("ccx", [10, 11, 12])
print(rt5.round(3).to_string(index=False))
for cells in ("pyramidal", "all"):
    P5 = POP[("ccx", cells, "code")]
    R5c = pd.DataFrame([session_rocs(P, cd_projections(P)[0], [10, 11, 12]) for P in P5])
    R2c = pd.DataFrame([session_rocs(P, cd_projections(P)[0], [4, 5, 6]) for P in POP[("exp2", cells, "code")]])
    ta = stats.ttest_rel(R5c[10], R5c[0])
    tb = stats.ttest_ind(R5c[10], R2c[4])
    print(f"{cells}: callosotomy sessions {len(R5c)}; ipsilateral ROC {R5c[10].mean():.3f} against callosotomy control {R5c[0].mean():.3f} (paired t = {ta.statistic:.2f}, p = {ta.pvalue:.3g}); "
          f"against Experiment 2 ipsilateral {R2c[4].mean():.3f} (t = {tb.statistic:.2f}, p = {tb.pvalue:.3g})")
print("Paper: 16 callosotomy sessions in Fig. 5f.")

**Result: reproduced.** In the callosotomy sessions the population rate is back to 1.05 of control just before the cue after ipsilateral photoinhibition, while ipsilateral selectivity stays 67% below control 0.2 s before the cue and recovers in under 1% of resamples. Contralateral selectivity recovers in 297 ms (code reading, 81% of resamples), within 217 +- 228 ms, and bilateral selectivity does not recover (0 of 1000 resamples). Across the 64 readings ipsilateral selectivity reaches 80% in at most 9% of resamples and bilateral in at most 0.2% (one or two of 1000 resamples in seven readings, none in the others), and 37 readings also put the contralateral time within tolerance. The selection has 125 neurons against 129. The session rule gives 16 callosotomy sessions, as Fig. 5f, and the ipsilateral cd ROC at the end of the delay is 0.566, lower than the callosotomy control (0.718, paired p = 0.006) and the Experiment 2 ipsilateral ROC (0.706, p = 0.003).

## ed9f-reaction-time

Claim: reaction times are faster on trials whose cd trajectory lies far from the decision
boundary at the go cue. Sessions of fig3-cd-roc whose lick trace shows licks (data check
above); reaction time the first lick after the cue (criterion's reading), relative to the
session mean, lick right and lick left trials pooled, correct and error trials; Spearman
correlation with the absolute projection in the last 400 ms of the delay, negative with p < 0.05
(criterion). Trials (a) control test trials, (b) all conditions.

In [ ]:
print("Sessions whose lick trace shows licks, by experiment:")
print(pd.crosstab(lk.experiment, usable.map({True: "lick-like", False: "noise-like"})).to_string())
ok_sessions = set(lk[usable].session)
print("Time series channels of the Experiment 2 sessions:", sorted({tuple(SD[n]["ts_channels"]) for n in ses[ses.experiment == "exp2"].session}))
print("Experiment 2 lick traces, range (99.9th percentile minus median) per session:", sorted(lk[lk.experiment == "exp2"].trace_range.round(4).tolist()))
for exp, conds in (("exp2", (4, 5, 6)), ("exp3", (7, 8, 9)), ("exp2b", (4, 5, 7)), ("ccx", (10, 11, 12))):
    rows = []
    for P in POP[(exp, "pyramidal", "code")]:
        if P["session"] not in ok_sessions:
            continue
        s = SD[P["session"]]
        proj, _, _ = cd_projections(P)
        rt = (s["first_lick"] - s["cue"])[P["trials"]]
        for c in (0,) + conds:
            idx = np.r_[P["test"][1], P["test"][-1]] if c == 0 else np.r_[P["groups"][(c, 1)], P["groups"][(c, -1)]]
            for i in idx:
                if np.isfinite(rt[i]):
                    rows.append({"session": P["session"], "mouse": P["mouse"], "condition": c, "rt": rt[i], "dist": abs(proj[i, I_LATE])})
    print(f"\n{exp}: population sessions with lick-like traces: {len({r['session'] for r in rows})}")
    if not rows:
        continue
    d9 = pd.DataFrame(rows)
    d9["rt_rel"] = d9.rt - d9.groupby("session").rt.transform("mean")
    print(f"  {d9.mouse.nunique()} mice, {len(d9)} trials; reaction time (s) median {d9.rt.median():.3f}, range {d9.rt.min():.3f} to {d9.rt.max():.3f}")
    for lab, m in (("(a) control", d9.condition == 0), ("(b) all conditions", d9.condition >= 0)):
        r = stats.spearmanr(d9[m].dist, d9[m].rt_rel)
        print(f"  {lab}: n = {int(m.sum())}, Spearman r = {r.statistic:.3f}, p = {r.pvalue:.3g}")

**Result: blocked.** None of the 26 Experiment 2 sessions has a lick trace with detectable licks (trace range 0.0013 to 0.0019; their only channels are lick_trace, aom_input_trace and laser_power), so the claim's reaction times cannot be measured. As variants on other sessions whose traces show licks: in the 4 Experiment 3 population sessions, reaction time does not correlate with the distance from the boundary on control trials (Spearman r = -0.019, p = 0.84) and correlates negatively over all conditions (r = -0.153, p = 0.001); in the 5 sessions with types 4, 5 and 7, r = 0.017 (p = 0.86) and 0.029 (p = 0.55); in the 15 callosotomy sessions, r = -0.025 (p = 0.66) and -0.025 (p = 0.38).

## models-monolithic-fail

Claim: monolithic network models (fixed-point attractor, integrators, trained recurrent
networks) do not recover their unperturbed trajectory after transient silencing of half of their
neurons. A simulation that uses no recorded data; parameters from the Methods and the ED Fig. 1
legend, open details as stated in `models.py`. Criterion: in each model, the output at t = 2 s
after perturbation differs from the unperturbed output by more than 20% of the unperturbed
selectivity in at least two of three draws. Run here: the simple integrator (three draws of its
connectivity), the monolithic attractor (one module of Fig. 6b; either neuron silenced; noise
realizations as draws) and the FORCE-trained network (three draws). Not run: the integrator with
corrective feedback (ED Fig. 1g), whose weights the paper does not give, and the tamed-chaos
network (ED Fig. 1i), whose pruning threshold and retuning the paper does not give.

In [ ]:
import models

rng_m = np.random.default_rng(2016)
rows = []
for d in range(3):
    u = rng_m.normal(size=100)
    u /= np.linalg.norm(u)
    sil = rng_m.choice(100, 50, replace=False)
    o = {tr: models.run_simple_integrator(tr, None, u)[-1] for tr in (1, -1)}
    p = {tr: models.run_simple_integrator(tr, sil, u)[-1] for tr in (1, -1)}
    sel = o[1] - o[-1]
    rows.append({"model": "simple integrator", "draw": d, "unperturbed_out": o[1], "perturbed_out": p[1],
                 "deviation_over_selectivity": max(abs(p[1] - o[1]), abs(p[-1] - o[-1])) / abs(sel)})
W1 = models.attractor_weights()[:2, :2]
for d in range(3):
    for silenced in ([0], [1]):
        seed = int(rng_m.integers(1e9))
        o = {tr: models.run_attractor(tr, None, np.random.default_rng(seed + tr), W=W1)[-1] for tr in (1, -1)}
        p = {tr: models.run_attractor(tr, silenced, np.random.default_rng(seed + tr), W=W1)[-1] for tr in (1, -1)}
        sel = (o[1][0] - o[1][1]) - (o[-1][0] - o[-1][1])
        dev = max(abs((p[tr][0] - p[tr][1]) - (o[tr][0] - o[tr][1])) for tr in (1, -1))
        rows.append({"model": f"monolithic attractor, neuron {silenced[0] + 1} silenced", "draw": d, "unperturbed_out": o[1][0] - o[1][1],
                     "perturbed_out": p[1][0] - p[1][1], "deviation_over_selectivity": dev / abs(sel)})
t_f = time.time()
for d in range(3):
    net = models.Force(np.random.default_rng(300 + d))
    net.train(30)
    sil = np.random.default_rng(400 + d).choice(400, 200, replace=False)
    o = {tr: net.run(tr) for tr in (1, -1)}
    p = {tr: net.run(tr, sil) for tr in (1, -1)}
    sel = o[1][-1] - o[-1][-1]
    rows.append({"model": "FORCE network", "draw": d, "unperturbed_out": o[1][-1], "perturbed_out": p[1][-1],
                 "deviation_over_selectivity": max(abs(p[tr][-1] - o[tr][-1]) for tr in (1, -1)) / abs(sel),
                 "train_error_rms": float(np.sqrt(np.mean((o[1] - net.target(np.arange(2000) * 0.001, 1)) ** 2)))})
print(f"FORCE networks trained and run in {time.time() - t_f:.0f} s")
mono = pd.DataFrame(rows)
mono["not_recovered"] = mono.deviation_over_selectivity > 0.2
print(mono.round(3).to_string(index=False))
print("\nDraws not recovered (deviation > 20% of selectivity), per model:")
print(mono.groupby("model").not_recovered.agg(["sum", "size"]).to_string())

**Result: partially reproduced.** Each of the three simulated monolithic models fails to recover in all three draws: the simple integrator ends 21% of its selectivity from the unperturbed output, the single-module attractor switches to the other fixed point (deviation 1.02 to 1.09 of its selectivity), and the FORCE-trained networks end 25% to 87% away. The corrective-feedback integrator and the tamed-chaos network were not simulated, as the paper does not give their weights or retuning. The criterion has no partial rule; the outcome is partial because two of the five models could not be tested. The simple integrator's deviation (0.212 against the threshold of 0.2) depends on the assumed rank-one matrix.

## fig6-modular-models-robust

Claim: modular networks of two coupled modules restore activity after transient silencing of
one module but drift to a random fixed point when both are silenced. Modular attractor and
modular integrator with the Methods' parameters (`models.py`); the modular FORCE network is not
run, as its training of the inter-module weights is not specified. Criterion: for the attractor
and the integrator, the silenced module returns within 10% of its unperturbed state by t = 2 s in
at least 90% of 100 noisy runs, and after bilateral silencing the attractor ends at the wrong
fixed point in 25% to 75% of runs. The module's state is read as its selectivity (right minus
left preferring neuron), compared with the same noise realization without silencing; the fixed
point at t = 2 s by the sign of that selectivity. The integrator has no noise, so one run per
trial type stands for all. Bilateral silencing of the attractor is also run to 4 s, so that every
run settles at a fixed point (verifier, round 1).

In [ ]:
rows = []
for noise in ("step", "time"):
    for tr in (1, -1):
        rec, wrong, wrong_ctrl, undecided = [], [], [], []
        for i in range(100):
            seed = 10_000 * (1 if noise == "step" else 2) + 100 * (tr + 1) + i
            o = models.run_attractor(tr, None, np.random.default_rng(seed), noise=noise)
            p = models.run_attractor(tr, [0, 1], np.random.default_rng(seed), noise=noise)
            b = models.run_attractor(tr, [0, 1, 2, 3], np.random.default_rng(seed), noise=noise)
            so, sp, sb = o[-1, 0] - o[-1, 1], p[-1, 0] - p[-1, 1], b[-1, 0] - b[-1, 1]
            rec.append(abs(sp - so) <= 0.1 * abs(so))
            wrong.append(np.sign(sb) != tr)
            wrong_ctrl.append(np.sign(so) != tr)
            undecided.append(abs(sb) < 0.5 * abs(so))
        rows.append({"model": "modular attractor", "noise": noise, "trial": tr, "runs": 100,
                     "unilateral_recovered": np.mean(rec), "bilateral_wrong_fixed_point": np.mean(wrong),
                     "bilateral_not_settled_at_2s": np.mean(undecided), "control_wrong": np.mean(wrong_ctrl)})
for tr in (1, -1):
    o = models.run_integrator(tr, None)
    p = models.run_integrator(tr, [0, 1, 2, 3])
    b = models.run_integrator(tr, list(range(8)))
    so, sp, sb = o[-1, 0] - o[-1, 1], p[-1, 0] - p[-1, 1], b[-1, 0] - b[-1, 1]
    rows.append({"model": "modular integrator", "noise": "none", "trial": tr, "runs": 1,
                 "unilateral_recovered": float(abs(sp - so) <= 0.1 * abs(so)), "bilateral_wrong_fixed_point": float(np.sign(sb) != tr),
                 "selectivity_unperturbed": so, "selectivity_after_unilateral": sp, "selectivity_after_bilateral": sb})
m6 = pd.DataFrame(rows)
print(m6.round(3).to_string(index=False))
# Bilateral silencing run to 4 s, so that the attractor settles (noise drawn each step).
rows = []
for tr in (1, -1):
    end = []
    for i in range(100):
        seed = 30_000 + 100 * (tr + 1) + i
        o = models.run_attractor(tr, None, np.random.default_rng(seed), t_end=4.0)
        b = models.run_attractor(tr, [0, 1, 2, 3], np.random.default_rng(seed), t_end=4.0)
        so = o[-1, 0] - o[-1, 1]
        end.append({"at_2s": b[1999, 0] - b[1999, 1], "at_4s": b[-1, 0] - b[-1, 1], "so": so})
    e = pd.DataFrame(end)
    settled = e.at_4s.abs() >= 0.5 * e.so.abs()
    rows.append({"trial": tr, "runs": len(e), "settled_at_2s": float((e.at_2s.abs() >= 0.5 * e.so.abs()).mean()), "settled_at_4s": float(settled.mean()),
                 "right_fixed_point_at_4s": float((settled & (np.sign(e.at_4s) == 1)).mean()), "left_fixed_point_at_4s": float((settled & (np.sign(e.at_4s) == -1)).mean()),
                 "wrong_fixed_point_at_4s": float((settled & (np.sign(e.at_4s) != tr)).mean()),
                 "wrong_among_settled": float((np.sign(e.at_4s[settled]) != tr).mean()) if settled.any() else np.nan})
print("\nModular attractor, bilateral silencing, run to 4 s (settled: |R - L| at least half the unperturbed value):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced.** With noise drawn each step, the modular attractor recovers after unilateral silencing in 100% and 98% of runs (lick right and lick left). After bilateral silencing it ends at the wrong fixed point at 2 s in 45% and 63% of runs, but 80% and 83% of those runs have not settled; run to 4 s, every run settles, at the wrong fixed point in 47% and 55%, within 25% to 75%. With noise of the same intensity per unit time the unperturbed attractor already ends at the wrong fixed point in 45% to 49% of runs. The modular integrator with the printed matrix recovers on lick right trials (selectivity 13.6 against 14.4) but not on lick left trials (-7.2 against -14.4). The attractor part of the criterion holds; the integrator part holds for one trial type, so the integrator recovers in 50% of trial types where the criterion needs 90% of runs. The criterion has no partial rule, so by the rule applied to the other criteria without one this result is not reproduced; the verifier agreed in its third round. Whether the printed matrix lacks part of the network of Fig. 6c is a question about the paper's methods and does not make the outcome partial.

In [ ]:
print(f"Notebook run time: {time.time() - started:.0f} s from the start of loading.")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| units-and-sessions | not reproduced (FS count 89 against 72; the other five counts exact) |
| ed1-selective-fraction | reproduced |
| fig1d-late-delay-ipsilateral-bias | blocked: data missing |
| fig1d-earlier-photoinhibition-minimal | reproduced |
| ed3-recovery-interval | blocked: data missing |
| ed2b-silencing | reproduced |
| ed2d-spatial-extent | blocked: data missing |
| ed2e-no-rebound | not reproduced (more neurons significantly excited than suppressed) |
| fig1f-rate-recovery | partially reproduced |
| fig1g-selectivity-recovery | partially reproduced (times match; 66 to 160 neurons against 55) |
| ed4-return-to-trajectory | not reproduced |
| ed5-photoactivation | blocked: data missing |
| fig2b-bilateral-behaviour | partially reproduced |
| ed6-bilateral-bias-inconsistent | reproduced (under reading b only) |
| fig2-bilateral-recovery | partially reproduced (ipsilateral recovery in 6 of 64 readings; bilateral rate 11% above control) |
| ed7-unilateral-4-spots-bilateral-1-spot | not reproduced (bilateral 1-spot selectivity recovers) |
| ed7e-size-and-roc | not reproduced (bilateral 1-spot ROC above tolerance) |
| fig3-cd-roc | reproduced |
| ed8-roc-during-photoinhibition | reproduced |
| ed9b-cd-stability | reproduced |
| ed8a-mode-variance | partially reproduced |
| fig3-cd-predicts-movement | reproduced |
| fig3-other-modes | not reproduced |
| ed9f-reaction-time | blocked: data missing (no licks in the Experiment 2 lick traces) |
| fig5b-callosotomy-behaviour | partially reproduced (after the cut only) |
| fig5-callosotomy-recovery | reproduced |
| ed10e-callosotomy-preparatory-activity | not reproduced (519 neurons against 496; fraction 14 points lower) |
| models-monolithic-fail | partially reproduced (three of five models simulated) |
| fig6-modular-models-robust | not reproduced (the integrator recovers on one trial type only) |

**How the analysis differs from the paper's.** Open method details are taken from the authors'
MATLAB scripts deposited in the data set: trial selection (correct and error trials, stable trials,
lick-early trials dropped), the preference screening, the recovery-time bootstrap, the neuron
selections and the population session rule. The scripts reproduce several of the paper's counts
exactly (168, 168 and 175 neurons in Fig. 1f; 16 sessions in Fig. 3 and Fig. 5f). The coding
direction is the mean difference of the Methods; the scripts' linear discriminant is shown as a
variant. Every reading of the preference screening and of the neuron selection (the scripts' or the
claims file's) is run for the recovery times, and a recovery time counts only when most bootstrap
resamples recover. The population session rule keeps groups of 5 trials, as the scripts do, which
gives the paper's 16 sessions; the Methods' "> 5 trials" gives 12. Rates and PSTHs are computed from
10 ms bins, and recovery times are resolved to 10 ms.
Behavioural claims use the recording sessions, as the behaviour-only sessions are not in the data
set. The network simulations use standard choices where the Methods leave details open
(`models.py`).

**What was found about the data set.** It holds what its description says: 91 sessions from 19
mice, 1012 single units in the 58 non-callosotomy sessions and 567 in the 33 callosotomy
sessions, with trial tables, photostimulus types whose laser timing matches their description,
and the gene modification of each mouse in the meta data files. The FS labels do not match the
paper's count (89 against 72) or the deposited waveforms' widths. The units have no `depth`
field, which the description lists. The lick trace shows no licks in 48 of 90 sessions, all 26
Experiment 2 sessions among them, so reaction times cannot be measured there. In four sessions
photoinhibition does not reduce activity, so the clock check by the laser fails there. Type 9 is
still described in the files as "Right ALM (ipsilateral)", which the FAQ says is contralateral.
One callosotomy session has no performing trials. The behaviour-only sessions, the photoactivation
data, the recordings in untrained mice and the simulations named in the title are not in the data
set.